# Donor Clustering Feature Build — v1

Produces one row per donor as of snapshot date `T`, built to the v1 clustering spec.

**Window policy.** Every clustering feature is sourced from `W = [T − 12 months, T)`.
Three declared exceptions: velocity (needs `W_prev`), tenure (needs lifetime), and
reactivation detection (needs history before `W_prev`).

**Roles.** Every field is tagged `CLUSTER`, `PROFILE`, `DIAG`, or `KEY`. All fields are
built and written out; the clustering script filters by role. `PROFILE` holds the
value/volume block that would otherwise reproduce the Hanover confound.

**What this notebook does not do.** No imputation, no scaling, no encoding. Those belong
in the clustering script — see the transform checklist in the final section.

---
## Section 0 — Configuration

In [1]:
from __future__ import annotations

import hashlib
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# --- paths -------------------------------------------------------------------
LOAD_PATH = Path("/Users/matt.fritz/Desktop/Behavioral Personas/SQL Donor Data")
WRITE_PATH = Path("/Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data")

FILES = {
    "dpr": "DonorProjectRecords_alldonors_from_last5yr.csv",
    "email": "Email Events 36mo.csv",
    "site": "Site Events FY24-26.csv",
    "monthly": "Monthly Donation Data All Time.csv",
    "share": "Share Events All Time.csv",
    "project_dates": "Project Dates FY22-26.csv",
    "zip_acs": "Merged_Zip_ACS_Demographics.csv",
}

# --- run parameters ----------------------------------------------------------
# Prefer a MONTH-START T. Monthly-grain sources (email, share) have their dates snapped to
# the 1st before the window filter, so a mid-month T silently drops the opening partial
# month for them while keeping it for gifts -- email exposure ends up understated relative
# to giving, and gift-based vs email-based month counts get different ceilings.
T = pd.Timestamp("2026-08-01")   # snapshot date

WINDOW_MONTHS = 24               # primary window W
PREGIFT_LOOKBACK_DAYS = 7        # decision-speed pre-gift window (inclusive of gift day)
PROJECT_LIFECYCLE_MONTHS = 4     # DonorsChoose projects are open for four calendar months
LATE_CYCLE_POSITION = 0.75        # final quarter of the project lifecycle
EXPIRING_SOON_DAYS = 7           # final seven calendar days through expiration date
MIN_LEVEL_PREVALENCE = 0.03      # share-vector level cutoff
MAX_LEVELS = 15                  # cap per share vector
FIXED_AMOUNT_MODAL_SHARE = 0.80  # is_fixed_amount_donor_12m threshold
RELIABLE_SHARE_MIN_GIFTS = 3     # shares_reliable_12m threshold
# Project lifecycle features now use `project_expiration_date` directly on DPR. The legacy
# `project_dates` input remains in the function contract for backward compatibility only.
LOAD_PROJECT_DATES = False

# --- monthly source contract -------------------------------------------------
# Empirically verified source grain:
#   one row = one recurring payment
#   recurring_donation_id = one recurring-giving relationship
# Do not infer alternate grains. If this contract changes, fail loudly and inspect the new
# source rather than guessing inside production feature code.
MONTHLY_RELATIONSHIP_ID = "recurring_donation_id"
MONTHLY_PAYMENT_DATE = "monthly_subscription_payment_date"
MONTHLY_PAYMENT_AMOUNT = "monthly_subscription_payment_amount"
MONTHLY_JOIN_DATE = "monthly_subscription_joined_date"
MONTHLY_RETIRED_DATE = "monthly_subscription_retired_date"          # QA only
MONTHLY_ACTIVE_LOOKBACK_DAYS = 30   # behavioral activity: recent observed payment before T
MONTHLY_REQUIRED_COLS = {
    "donor_id", MONTHLY_RELATIONSHIP_ID, MONTHLY_PAYMENT_DATE,
    MONTHLY_PAYMENT_AMOUNT, MONTHLY_JOIN_DATE,
}
# Tripwires against loading a truncated / wrong export. These are not modelling thresholds.
MIN_MONTHLY_ROWS = 1_000
MIN_MONTHLY_DONORS = 1_000

# --- performance -------------------------------------------------------------
# Restricting the event tables to the donor universe before feature work is loss-free: the
# universe is defined only from `dpr` and `monthly`, so email/site/share can never add a
# donor. On a 50k-donor universe against 84.5M email rows this is the single biggest win.
RESTRICT_TO_UNIVERSE = True
RESTRICT_COLUMNS = True          # read only the columns the builder uses
USE_SOURCE_CACHE = True          # cache a column-restricted binary copy of each CSV
CACHE_DIR = WRITE_PATH / "_source_cache"
RAISE_ON_UNUSABLE_SOURCE = False  # other sources only; monthly contract always fails fast
OUT_TAG = f"{T:%Y%m%d}_w{WINDOW_MONTHS}m"

# --- columns the builder actually reads ---------------------------------------
# Declared so the giant CSVs are not read in full. Non-monthly blocks handle optional
# columns gracefully; the monthly required columns are enforced by `validate_monthly()`.
USECOLS: dict[str, set[str]] = {
    "dpr": {
        "donor_id", "payment_date", "payment_amount", "donation_n", "teacher_id",
        "school_id", "project_category", "project_grade", "project_total_cost",
        "project_expiration_date",
        "project_got_fully_funded", "project_fully_funded", "gift_completed_project_funding",
        "school_is_historically_underrepresented_race", "school_is_low_income",
        "school_is_underserved_rural", "match_xyi_multiplier",
        "referral_source", "referral_medium", "payment_type", "daf_payment",
        "green_payment_amount", "is_green_payment", "gift_card_purchase",
        "payment_on_big_event", "optional_donation_rate", "donation_is_anonymous",
        "is_classroom_essentials_list", "donor_zip", "school_zip", "donor_lat_long",
        "school_lat_long", "is_teacher", "is_teacher_referred",
        "subscribed_to_marketing_emails", "major_gift_donor", "account_credit_balance",
        "gift_is_projects_first", "gift_is_projects_last",
        "teacher_lifetime_projects_fully_funded", "teacher_lifetime_donations",
    },
    "email": {"donor_id", "email_sent_month", "email_month_start", "email_sent_count",
              "email_open_count", "email_click_count"},
    "site": {"donor_id", "activity_date", "session_id", "session_duration_min",
             "cart_visits_day", "came_from_campaign", "project_page_visits_day",
             "teacher_page_visits_day", "search_visits_day", "device_type"},
    "monthly": {"donor_id", MONTHLY_RELATIONSHIP_ID, MONTHLY_PAYMENT_DATE,
                MONTHLY_PAYMENT_AMOUNT, MONTHLY_JOIN_DATE, MONTHLY_RETIRED_DATE},
    "share": {"donor_id", "share_sent_month", "share_month_start", "share_event_count"},
    # zip_acs is small and its ACS column names vary, so it is read whole
    "zip_acs": None,
}


# Bump when anything about how a source is READ changes -- aliases, blank-row cleanup,
# dtype coercion. The cache key includes it, so a logic change invalidates the cache even
# though the CSV has not moved. Without this, a cache written under an older column set is
# served silently and every downstream diagnosis is about the cache, not the file.
SOURCE_READ_VERSION = 6


def canonical_donor_id(s: pd.Series) -> pd.Series:
    """One canonical string form for `donor_id` across every source.

    A bare `astype(str)` is not enough: a numeric id column becomes float64 the moment any
    row is null, and `38877.0` will never match `38877`. Both sides must route through the
    same integer normalisation, or the cross-source join silently returns nothing.
    """
    if s.dtype == object or pd.api.types.is_string_dtype(s):
        txt = s.astype("string").str.strip()
        num = pd.to_numeric(txt, errors="coerce")
        # keep the text form for genuinely non-numeric ids, integer form for numeric ones
        return txt.where(num.isna(), num.astype("Int64").astype("string"))
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype("string")


def _cache_key(source: str, wanted: set[str] | None) -> str:
    """Cache filename stem, fingerprinted on the requested columns and read version."""
    cols = ",".join(sorted(wanted)) if wanted else "ALL"
    digest = hashlib.sha1(f"{SOURCE_READ_VERSION}|{cols}".encode()).hexdigest()[:10]
    return f"{source}.v{SOURCE_READ_VERSION}.{digest}"


def load_source(path: Path, source: str, restrict_columns: bool = RESTRICT_COLUMNS,
                cache_dir: Path | None = None, verbose: bool = True) -> pd.DataFrame:
    """Read one source CSV, column-restricted, cleaned, and binary-cached.

    The cache is invalidated by the CSV's modification time AND by the requested column set
    AND by `SOURCE_READ_VERSION`, so neither a new export nor a change to what the notebook
    asks for can be masked by a stale copy.
    """
    wanted = USECOLS.get(source) if restrict_columns else None
    # Parquet is portable across pandas versions; pickle is not, so its filename carries the
    # major version. A cache that cannot be read falls back to the CSV rather than raising.
    pdv = pd.__version__.split(".")[0]
    stem = _cache_key(source, wanted)
    candidates = ((cache_dir / f"{stem}.parquet", pd.read_parquet, "to_parquet"),
                  (cache_dir / f"{stem}.pd{pdv}.pkl", pd.read_pickle, "to_pickle")) \
        if cache_dir is not None else ()

    for cached, reader, _ in candidates:
        if cached.exists() and cached.stat().st_mtime >= path.stat().st_mtime:
            try:
                df = reader(cached)
            except Exception as exc:
                if verbose:
                    print(f"  {source:<14s} cache unreadable ({type(exc).__name__}); "
                          f"re-reading the CSV")
                continue
            # a hash cannot catch everything, so verify the frame covers the request
            if wanted and not (wanted & set(df.columns)) <= set(df.columns):
                continue
            if verbose:
                print(f"  {source:<14s} {len(df):>12,} rows x {df.shape[1]:>2} cols "
                      f"from cache {cached.name}")
            return df

    df = pd.read_csv(path, low_memory=False,
                     usecols=(lambda c: c in wanted) if wanted else None)

    # A spreadsheet that has opened a large CSV writes out its whole used range on save,
    # padding the file with entirely empty rows. Those inflate row counts, dilute every
    # completeness measure, and make a healthy column look 99.99% unpopulated.
    n_before = len(df)
    df = df.dropna(how="all")
    if len(df) < n_before:
        df = df.reset_index(drop=True)
        if verbose:
            print(f"  {source:<14s} dropped {n_before - len(df):,} entirely-blank rows "
                  f"({n_before:,} -> {len(df):,}) -- likely spreadsheet padding")

    if "donor_id" in df.columns:
        df["donor_id"] = canonical_donor_id(df["donor_id"])

    if wanted and verbose:
        missing = sorted(wanted - set(df.columns))
        if missing:
            print(f"  {source:<14s} not in this file: {missing}")

    for cached, _, writer in candidates:
        cached.parent.mkdir(parents=True, exist_ok=True)
        try:
            getattr(df, writer)(cached)
            break
        except Exception:
            continue
    return df


def report_donor_id_overlap(sources: dict[str, pd.DataFrame]) -> None:
    """Print donor_id dtype per source and the overlap with the project extract.

    Zero overlap between dpr and monthly cannot be a real fact at this scale, so it is
    called out rather than left to surface as a quiet "100% dropped" line later.
    """
    print("\ndonor_id across sources:")
    base = sources.get("dpr")
    base_ids = set(base["donor_id"].dropna()) if base is not None \
        and "donor_id" in base.columns else set()
    for name, df in sources.items():
        if df is None or "donor_id" not in df.columns:
            print(f"  {name:<14s} no donor_id column")
            continue
        ids = set(df["donor_id"].dropna())
        line = f"  {name:<14s} dtype={str(df['donor_id'].dtype):<8s} {len(ids):>9,} distinct"
        if name != "dpr" and base_ids:
            shared = len(ids & base_ids)
            line += f" | {shared:>9,} shared with dpr ({shared / max(len(ids), 1):.1%})"
            if shared == 0:
                line += "  <-- CHECK: no donor matches the project extract at all"
        print(line)

## Section 1 — Primitives

Four things here matter downstream. `safe_div` returns `NaN` on a zero denominator rather
than dividing by an epsilon — the `1e-6` pattern is what produced ratios near `1e6` that
would dominate any distance metric. `bounded_ratio` keeps velocity in `(−1, 1)`.
`share_where` computes a proportion over **observable** rows only, since a missing input
compares False and would otherwise be averaged in as a negative observation. And month
arithmetic is split into `completed_months` (whole months elapsed) and `elapsed_months`
(continuous, days / 30.4375) — subtracting year/month numbers counts calendar boundaries
crossed, so May 31 → Jun 1 reads as a month when one day has passed.

In [2]:
def safe_div(num: pd.Series, den: pd.Series, fill: float = np.nan) -> pd.Series:
    """Element-wise divide. Zero/NaN denominator -> `fill`. No epsilon fudge."""
    num, den = num.align(den)
    num = num.astype("float64")
    den = den.astype("float64")
    out = pd.Series(fill, index=num.index, dtype="float64")
    ok = den.notna() & (den != 0) & num.notna()
    out[ok] = num[ok] / den[ok]
    return out


def bounded_ratio(a: pd.Series, b: pd.Series, k: float = 1.0) -> pd.Series:
    """Signed relative change in (-1, 1). k damps the small-denominator case."""
    a, b = a.align(b)
    a = a.astype("float64").fillna(0.0)
    b = b.astype("float64").fillna(0.0)
    return (a - b) / (a + b + k)


TRUE_TOKENS = {"t", "true", "y", "yes", "1", "1.0"}


def to_flag(s: pd.Series) -> pd.Series:
    """Coerce bool / 0-1 / 't'-'f' / 'Y'-'N' to float 0.0-1.0."""
    if s.dtype == bool:
        return s.astype("float64")
    if pd.api.types.is_numeric_dtype(s):
        return (s.fillna(0) > 0).astype("float64")
    return s.astype("string").str.strip().str.lower().isin(TRUE_TOKENS).astype("float64")


def slug(x) -> str:
    """Level value -> safe column suffix."""
    s = re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower())
    return re.sub(r"_+", "_", s).strip("_") or "blank"


def dedupe(names: list[str]) -> list[str]:
    seen, out = {}, []
    for n in names:
        if n in seen:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            out.append(n)
    return out


def calendar_months_touched(start, end_exclusive) -> int:
    """Distinct calendar months intersected by `[start, end)`.

    This is the ceiling for a month count built from DAILY-grain data. A 12-month window
    that does not begin on the 1st touches 13 calendar months, so comparing such a count
    against `WINDOW_MONTHS` produces a false alarm.
    """
    a = pd.Timestamp(start)
    b = pd.Timestamp(end_exclusive) - pd.Timedelta(days=1)
    return (b.year - a.year) * 12 + (b.month - a.month) + 1


def month_starts_in(start, end_exclusive) -> int:
    """Number of month-start timestamps in `[start, end)`.

    This is the ceiling for a month count built from MONTHLY-grain data (email, share),
    whose dates are snapped to the 1st before the window filter is applied. It is one less
    than `calendar_months_touched` whenever the window opens mid-month.
    """
    lo = pd.Timestamp(start).normalize()
    hi = pd.Timestamp(end_exclusive) - pd.Timedelta(days=1)
    return 0 if hi < lo else len(pd.date_range(lo, hi, freq="MS"))


# Which ceiling applies to each month-count field. Declared explicitly rather than inferred
# from the name, so a new month-count field cannot silently acquire the wrong bound -- the
# QA check reports any month-count field missing from this map.
MONTH_COUNT_BASIS = {
    "n_active_months_12m": "daily",
    "monthly_active_months_12m": "daily",
    "email_active_months_12m": "monthly_grain",
    "email_months_with_opens_12m": "monthly_grain",
    "sharing_active_months_12m": "monthly_grain",
}


def completed_months(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """Whole months actually elapsed, integer-valued.

    Subtracting year/month numbers alone counts calendar BOUNDARIES CROSSED, not elapsed
    time: May 31 -> Jun 1 is one boundary but one day. Decrementing when the day-of-month
    has not yet come round gives the intuitive reading (May 31 -> Jun 30 is 0 months,
    May 31 -> Jul 1 is 1).
    """
    later, earlier = later.align(earlier)
    raw = ((later.dt.year - earlier.dt.year) * 12
           + (later.dt.month - earlier.dt.month)).astype("float64")
    return raw - (later.dt.day < earlier.dt.day).astype("float64")


DAYS_PER_MONTH = 30.4375   # mean Gregorian month


def elapsed_months(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """Continuous months as days / 30.4375. Used where durations are SUMMED, so that a
    25-day subscription episode contributes 0.8 rather than vanishing at 0."""
    later, earlier = later.align(earlier)
    return (later - earlier).dt.days.astype("float64") / DAYS_PER_MONTH


def empty_frame(index: pd.Index, cols: list[str]) -> pd.DataFrame:
    """All-NaN frame with the given columns. Used when a source table is unusable, so
    downstream column expectations hold regardless of input completeness."""
    return pd.DataFrame(np.nan, index=index, columns=cols, dtype="float64")

## Section 2 — Grouped aggregation helpers

In [3]:
def per_donor(df: pd.DataFrame, col: str, how: str, index: pd.Index) -> pd.Series:
    """Single aggregate of `col` per donor, reindexed. Missing column -> all-NaN.

    Dtype-preserving: numeric aggregates come back float64, datetime aggregates stay
    datetime (so `min`/`max` of a date column works through the same helper).
    """
    if df.empty or col not in df.columns:
        return pd.Series(np.nan, index=index, dtype="float64")
    s = df.groupby("donor_id", observed=True)[col].agg(how).reindex(index)
    return s.astype("float64") if pd.api.types.is_numeric_dtype(s) else s


def count_per_donor(df: pd.DataFrame, index: pd.Index, fill: float = np.nan) -> pd.Series:
    if df.empty:
        return pd.Series(fill, index=index, dtype="float64")
    return (df.groupby("donor_id", observed=True).size()
            .reindex(index).astype("float64").fillna(fill))


def share_flag(df: pd.DataFrame, col: str, index: pd.Index,
               weight: str | None = None) -> pd.Series:
    """Share of a donor's rows (or of their dollars, if `weight`) where `col` is true."""
    if df.empty or col not in df.columns:
        return pd.Series(np.nan, index=index, dtype="float64")
    f = to_flag(df[col])
    g = df["donor_id"]
    if weight is None:
        return f.groupby(g, observed=True).mean().reindex(index).astype("float64")
    w = df[weight].astype("float64")
    return safe_div((f * w).groupby(g, observed=True).sum(),
                    w.groupby(g, observed=True).sum()).reindex(index)


def share_expr(mask: pd.Series, df: pd.DataFrame, index: pd.Index) -> pd.Series:
    """Share of a donor's rows satisfying an already-computed boolean mask."""
    if df.empty:
        return pd.Series(np.nan, index=index, dtype="float64")
    return (mask.astype("float64").groupby(df["donor_id"], observed=True)
            .mean().reindex(index).astype("float64"))


def share_where(mask: pd.Series, observed: pd.Series, df: pd.DataFrame,
                index: pd.Index) -> pd.Series:
    """Share of OBSERVABLE rows satisfying `mask`.

    Rows where the input is missing leave the denominator entirely rather than counting as
    False. `share_expr` would score a gift with no coordinates as "not local", so a donor
    with one 1-mile gift and one unlocatable gift reads 0.5 when the answer among
    measurable gifts is 1.0.
    """
    if df.empty:
        return pd.Series(np.nan, index=index, dtype="float64")
    g = df["donor_id"]
    obs = observed.astype("boolean").fillna(False).astype("float64")
    hit = mask.astype("boolean").fillna(False).astype("float64") * obs
    return safe_div(hit.groupby(g, observed=True).sum(),
                    obs.groupby(g, observed=True).sum()).reindex(index)


def coverage(observed: pd.Series, df: pd.DataFrame, index: pd.Index) -> pd.Series:
    """Share of a donor's rows where the input was observable. Companion to `share_where`
    -- a share among 1 of 20 gifts should not be read like a share among 20 of 20."""
    return share_expr(observed.astype("boolean").fillna(False), df, index)


def fill_zero_where_observed(s: pd.Series, observed_ids, index: pd.Index) -> pd.Series:
    """0 for donors present in a source but with no qualifying rows; NaN for donors absent
    from that source. "Never opened an email we sent" is zero engagement, not unknown."""
    out = s.reindex(index).astype("float64")
    seen = pd.Series(index.isin(observed_ids), index=index)
    return out.mask(seen & out.isna(), 0.0)


def nunique_where_observed(values: pd.Series, donor_ids: pd.Series,
                           index: pd.Index) -> pd.Series:
    """Distinct observed values per donor. No observed value -> NaN, not zero."""
    seen = values.notna()
    if not seen.any():
        return pd.Series(np.nan, index=index, dtype="float64")
    return (values.loc[seen].groupby(donor_ids.loc[seen], observed=True).nunique()
            .reindex(index).astype("float64"))


def sort_gifts(df: pd.DataFrame) -> pd.DataFrame:
    """Canonical gift ordering. `donation_n` breaks payment_date ties so that "first" and
    "latest" gift attributes do not depend on source row order."""
    keys = ["donor_id", "payment_date"] + (["donation_n"] if "donation_n" in df.columns else [])
    return df.sort_values(keys, kind="mergesort")


def latest_row_value(df: pd.DataFrame, col: str, index: pd.Index) -> pd.Series:
    """Value of `col` on each donor's LAST row -- NaN if it is null there.

    Distinct from `groupby.last()`, which returns the last NON-NULL value and would report
    a stale balance from an earlier gift as though it were current.
    """
    if df.empty or col not in df.columns:
        return pd.Series(np.nan, index=index, dtype="float64")
    last = sort_gifts(df).groupby("donor_id", observed=True).tail(1)
    s = last.set_index("donor_id")[col].reindex(index)
    return s.astype("float64") if pd.api.types.is_numeric_dtype(s) else s


def group_entropy(df: pd.DataFrame, cat_col: str, index: pd.Index,
                  normalize: bool = False, n_levels: int | None = None) -> pd.Series:
    """Shannon entropy (nats) of a donor's distribution over `cat_col`.

    `normalize=True` divides by `log(min(n_gifts, n_levels))` -- the true ceiling, which is
    bounded by BOTH how many gifts the donor made and how many levels exist. Dividing by
    `log(n_gifts)` alone (an earlier version of this function) penalises frequency on any
    bounded family: a donor with 30 gifts spread evenly across 4 grades would score 0.41
    while a donor with 4 gifts across the same 4 grades scored 1.0.

    `n_levels` defaults to the global distinct count of `cat_col`. For unbounded families
    (teacher_id, school_zip) that count is large, so `min()` collapses to `n_gifts` and the
    behaviour is unchanged -- one code path is correct for bounded and unbounded alike.
    """
    if df.empty or cat_col not in df.columns:
        return pd.Series(np.nan, index=index, dtype="float64")
    c = (df.groupby(["donor_id", cat_col], observed=True).size()
         .rename("n").reset_index())
    c["tot"] = c.groupby("donor_id", observed=True)["n"].transform("sum")
    c["p"] = c["n"] / c["tot"]
    c["term"] = -c["p"] * np.log(c["p"])
    ent = c.groupby("donor_id", observed=True)["term"].sum()
    if normalize:
        K = int(n_levels) if n_levels is not None else int(df[cat_col].nunique(dropna=True))
        n = c.groupby("donor_id", observed=True)["tot"].first().astype("float64")
        cap = np.minimum(n, float(max(K, 1)))
        ent = safe_div(ent, pd.Series(np.log(cap), index=n.index))
        ent[cap <= 1] = np.nan
    return ent.reindex(index).astype("float64")


def pick_levels(donor_ids: pd.Series, values: pd.Series,
                min_prevalence: float = MIN_LEVEL_PREVALENCE,
                max_levels: int = MAX_LEVELS) -> list[str]:
    """Levels to keep as their own share column, thresholded on **donor** prevalence:
    the fraction of donors who used the level at least once.

    Gift prevalence (`value_counts` over rows) is the wrong statistic for segmentation --
    twenty heavy donors giving fifty gifts each to one category makes that category look
    like 8% of gifts while covering 0.4% of donors. It would earn its own dimension and
    then behave as a near-constant column.
    """
    v = values.astype("string")
    keep_rows = v.notna()
    if not keep_rows.any():
        return []
    n_donors = donor_ids[keep_rows].nunique()
    if n_donors == 0:
        return []
    prev = (donor_ids[keep_rows].groupby(v[keep_rows], observed=True).nunique()
            / n_donors).sort_values(ascending=False)
    return [str(x) for x in prev[prev >= min_prevalence].index[:max_levels]]


def level_report(donor_ids: pd.Series, values: pd.Series, kept: list[str],
                 amounts: pd.Series | None = None) -> pd.DataFrame:
    """Donor prevalence, gift share and dollar share per level, flagging what gets folded
    into `_other`. Thresholding on donors means a rare-but-valuable level can disappear --
    this makes that visible rather than silent.
    """
    v = values.astype("string")
    m = v.notna()
    if not m.any():
        return pd.DataFrame(columns=["donor_prevalence", "gift_share", "amount_share", "kept"])
    n_donors = donor_ids[m].nunique()
    rep = pd.DataFrame({
        "donor_prevalence": donor_ids[m].groupby(v[m], observed=True).nunique() / n_donors,
        "gift_share": v[m].value_counts(normalize=True),
    })
    if amounts is not None:
        a = amounts[m].astype("float64")
        rep["amount_share"] = a.groupby(v[m], observed=True).sum() / a.sum()
    else:
        rep["amount_share"] = np.nan
    rep["kept"] = rep.index.isin(kept)
    return rep.sort_values("donor_prevalence", ascending=False).round(4)


def share_matrix(df: pd.DataFrame, cat_col: str, levels: list[str], prefix: str,
                 suffix: str, index: pd.Index, weight: str | None = None,
                 fixed_levels: bool = False) -> pd.DataFrame:
    """Per-donor share vector over `cat_col`. Rows sum to 1.0 for donors with any data.

    Unlisted levels -> `_other`; nulls -> `_unknown`. When `fixed_levels` is True the
    `_other`/`_unknown` buckets are omitted (the level list is already exhaustive).
    """
    cols_expected = list(levels) + ([] if fixed_levels else ["other", "unknown"])
    names = dedupe([f"{prefix}{slug(c)}{suffix}" for c in cols_expected])
    if df.empty or cat_col not in df.columns:
        return empty_frame(index, names)

    d = pd.DataFrame({
        "donor_id": df["donor_id"].to_numpy(),
        "_w": (df[weight].astype("float64").to_numpy() if weight else 1.0),
    })
    raw = df[cat_col].astype("string")
    lvl = np.where(raw.isna(), "unknown", np.where(raw.isin(levels), raw, "other"))
    d["_lvl"] = lvl

    piv = d.pivot_table(index="donor_id", columns="_lvl", values="_w",
                        aggfunc="sum", fill_value=0.0)
    piv = piv.reindex(columns=cols_expected, fill_value=0.0)
    tot = piv.sum(axis=1)
    piv = piv.div(tot.where(tot > 0), axis=0)
    piv.columns = names
    return piv.reindex(index).astype("float64")

## Section 3 — Loading and window slicing

`prepare()` normalises dates, derives `distance_mi` once, and cuts every window a
feature block needs. Doing the slicing in one place is what keeps the window policy
actually enforced rather than merely documented.

It also grades each date column and records the result in `S["parse_log"]`. A date column
can be absent, empty, unparseable, or implausible; those are different source failures and
are reported separately. Excel serial dates receive one guarded recovery attempt before a
column is rejected.

The monthly source is deliberately stricter. Its grain is now a fixed, empirically verified
contract: **one row is one recurring payment and `recurring_donation_id` identifies the
recurring relationship**. `validate_monthly()` checks the required fields, source volume,
relationship ownership, and active-flag consistency and raises immediately if the contract
changes. The feature code therefore does not contain grain inference or retirement-based
episode reconstruction.

Monthly windows are simple:

- `monthly_pre_T`: payment rows with `payment_date < T`;
- `monthly_W`: payment rows with `W_start <= payment_date < T`;
- `monthly_relationships`: one row per donor + recurring relationship observed by T.

`monthly_subscription_retired_date` is parsed for QA only. It is not used to end a
relationship because the source contains material payment-after-retirement contradictions.


In [4]:
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = (np.radians(np.asarray(x, dtype="float64"))
                              for x in (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# --- date parsing -----------------------------------------------------------------
# A date column can fail in four distinct ways, and they need different treatment:
#   ABSENT      the column is not in the extract at all
#   EMPTY       the column exists but is (almost) never populated
#   UNPARSEABLE the values present are not dates
#   IMPLAUSIBLE the values parse but land outside any sensible range
#
# The last one is the sneakiest. Excel writes dates as days since 1899-12-30, so a
# spreadsheet round-trip yields serial integers, which read as int64 and which
# `pd.to_datetime` then treats as NANOSECONDS -- parsing "succeeds" at 100% and puts every
# row in January 1970. So quality is judged on plausibility, not on parse success, and a
# guarded Excel-epoch fallback recovers the column when that is what happened.
#
# EMPTY and ABSENT must be separated from UNPARSEABLE because quality is measured over
# values PRESENT: a column with one populated row out of a million scores 100% on the
# values it has, and would otherwise be declared healthy.
EXCEL_EPOCH = pd.Timestamp("1899-12-30")
EXCEL_SERIAL_MIN, EXCEL_SERIAL_MAX = 20_000, 80_000     # ~1954 to ~2119
PLAUSIBLE_DATE_MIN = pd.Timestamp("1995-01-01")         # predates DonorsChoose entirely
PLAUSIBLE_DATE_MAX = pd.Timestamp("2100-01-01")
MIN_DATE_QUALITY = 0.5        # share of PRESENT values that must be plausible dates
MIN_DATE_COMPLETENESS = 0.5   # share of ROWS that must be populated at all

# Per-column declarations. `gates` controls whether the date participates in the generic
# source-health banner. Monthly has an additional fail-fast source contract below.
CAP_PRESENCE = "presence"
DATE_COLUMN_DEFAULTS = {"gates": CAP_PRESENCE, "blank_is_meaningful": False,
                        "month_string": False}
DATE_COLUMN_SPEC: dict[str, dict[str, dict]] = {
    "dpr": {
        "payment_date": {},
        # Parsed and quality-reported, but its completeness must not invalidate all DPR features.
        "project_expiration_date": {"gates": None},
    },
    "monthly": {
        MONTHLY_PAYMENT_DATE: {},
        MONTHLY_JOIN_DATE: {},
        # retirement is intentionally QA-only; blank is common and meaningful
        MONTHLY_RETIRED_DATE: {"gates": None, "blank_is_meaningful": True},
    },
    "site": {"activity_date": {}},
    "email": {"email_month_start": {"month_string": True}},
    "share": {"share_month_start": {"month_string": True}},
}


def _date_quality(parsed: pd.Series, n_present: int) -> tuple[pd.Series, float]:
    """(mask of plausible dates, share of present values that are plausible)."""
    ok = parsed.notna() & parsed.between(PLAUSIBLE_DATE_MIN, PLAUSIBLE_DATE_MAX)
    return ok, (int(ok.sum()) / n_present) if n_present else 0.0


def _to_dt(df: pd.DataFrame, col: str, source: str, log: list,
           month_string: bool = False, gates: str | None = CAP_PRESENCE,
           blank_is_meaningful: bool = False) -> None:
    """Coerce a column to datetime64 in place and record how it went.

    An absent column is logged rather than skipped silently: a missing `charge_date` empties
    every window built from it, and that must not look like "no charges occurred".
    """
    n_rows = len(df)
    rec = {"source": source, "column": col, "n_rows": n_rows, "gates": gates}

    if col not in df.columns:
        log.append({**rec, "present": False, "n_present": 0, "completeness": 0.0,
                    "n_usable": 0, "date_quality": 0.0, "strategy": "ABSENT",
                    "sample_bad": [], "usable": False, "verdict": "ABSENT"})
        return

    n_present = int(df[col].notna().sum())

    if pd.api.types.is_datetime64_any_dtype(df[col]):
        parsed, strategy = df[col], "already datetime"
        ok, quality = _date_quality(parsed, n_present)
    else:
        parsed = pd.to_datetime(df[col], errors="coerce")
        strategy = "to_datetime"
        ok, quality = _date_quality(parsed, n_present)
        if quality < MIN_DATE_QUALITY:
            num = pd.to_numeric(df[col], errors="coerce")
            alt = pd.to_datetime(num.where(num.between(EXCEL_SERIAL_MIN, EXCEL_SERIAL_MAX)),
                                 unit="D", origin=EXCEL_EPOCH, errors="coerce")
            alt_ok, alt_quality = _date_quality(alt, n_present)
            if alt_quality > quality:
                parsed, strategy = alt, "excel serial"
                ok, quality = alt_ok, alt_quality

    sample = []
    if n_present:
        bad = df[col].where(~ok & df[col].notna()).dropna()
        if not bad.empty:
            sample = [str(v) for v in bad.unique()[:3]]

    # Implausible values are discarded rather than carried: a 1970 timestamp would silently
    # place a row outside every window instead of announcing itself.
    parsed = parsed.where(ok)
    df[col] = (parsed.dt.to_period("M").dt.to_timestamp() if month_string
               else parsed.dt.normalize())

    completeness = (n_present / n_rows) if n_rows else 0.0
    if n_present == 0:
        # Nothing present to judge. For a column where blank is itself information
        # (`retired_date` on an active subscription) that is expected, not a defect.
        complete_ok = quality_ok = blank_is_meaningful
        verdict = "ALL BLANK" if blank_is_meaningful else "EMPTY"
    else:
        complete_ok = blank_is_meaningful or completeness >= MIN_DATE_COMPLETENESS
        quality_ok = quality >= MIN_DATE_QUALITY
        verdict = ("OK" if (complete_ok and quality_ok)
                   else "NEARLY EMPTY" if not complete_ok
                   else "UNPARSEABLE")
    log.append({**rec, "present": True, "n_present": n_present,
                "completeness": completeness, "n_usable": int(ok.sum()),
                "date_quality": quality, "strategy": strategy, "sample_bad": sample,
                "usable": bool(complete_ok and quality_ok), "verdict": verdict})


def _parse_dates(df: pd.DataFrame, source: str, log: list) -> None:
    """Parse every declared date column for one source, driven by `DATE_COLUMN_SPEC` so the
    declarations and the calls cannot drift apart."""
    for col, opts in DATE_COLUMN_SPEC.get(source, {}).items():
        cfg = {**DATE_COLUMN_DEFAULTS, **opts}
        _to_dt(df, col, source=source, log=log, month_string=cfg["month_string"],
               gates=cfg["gates"], blank_is_meaningful=cfg["blank_is_meaningful"])


def _split_latlon(s: pd.Series) -> tuple[pd.Series, pd.Series]:
    parts = s.astype("string").str.split(",", n=1, expand=True)
    if parts.shape[1] < 2:
        nan = pd.Series(np.nan, index=s.index, dtype="float64")
        return nan, nan
    return (pd.to_numeric(parts[0], errors="coerce"),
            pd.to_numeric(parts[1], errors="coerce"))



def validate_monthly(monthly: pd.DataFrame) -> dict:
    """Validate the verified monthly-payment contract and return compact QA stats.

    The feature builder should never guess this source's grain again. A contract failure is
    a source problem, so raise here rather than manufacture plausible-looking donor zeros.
    """
    missing = sorted(MONTHLY_REQUIRED_COLS - set(monthly.columns))
    if missing:
        raise ValueError(f"monthly source is missing required columns: {missing}")
    if len(monthly) < MIN_MONTHLY_ROWS:
        raise ValueError(
            f"monthly source has only {len(monthly):,} rows; expected at least "
            f"{MIN_MONTHLY_ROWS:,}. Check that the full export was loaded.")

    n_donors = int(monthly["donor_id"].nunique(dropna=True))
    if n_donors < MIN_MONTHLY_DONORS:
        raise ValueError(
            f"monthly source has only {n_donors:,} distinct donors; expected at least "
            f"{MIN_MONTHLY_DONORS:,}. Check that the full export was loaded.")

    required_nonnull = ["donor_id", MONTHLY_RELATIONSHIP_ID, MONTHLY_PAYMENT_DATE,
                        MONTHLY_PAYMENT_AMOUNT, MONTHLY_JOIN_DATE]
    nulls = {c: int(monthly[c].isna().sum()) for c in required_nonnull}
    bad_nulls = {c: n for c, n in nulls.items() if n}
    if bad_nulls:
        raise ValueError(f"monthly required fields contain nulls: {bad_nulls}")

    # Relationship IDs are identifiers, not measures. Canonicalise numeric-looking values so
    # `1001` and `1001.0` cannot split one relationship after a dtype change.
    monthly[MONTHLY_RELATIONSHIP_ID] = canonical_donor_id(monthly[MONTHLY_RELATIONSHIP_ID])

    amount = pd.to_numeric(monthly[MONTHLY_PAYMENT_AMOUNT], errors="coerce")
    if amount.isna().any():
        raise ValueError(
            f"monthly payment amount has {int(amount.isna().sum()):,} non-numeric values")
    monthly[MONTHLY_PAYMENT_AMOUNT] = amount.astype("float64")

    owner_counts = monthly.groupby(MONTHLY_RELATIONSHIP_ID, observed=True)["donor_id"].nunique()
    if (owner_counts > 1).any():
        raise ValueError(
            f"{int((owner_counts > 1).sum()):,} recurring_donation_id values map to "
            "more than one donor")

    join_counts = monthly.groupby(MONTHLY_RELATIONSHIP_ID, observed=True)[MONTHLY_JOIN_DATE].nunique()
    join_conflicts = int((join_counts > 1).sum())

    relationships = monthly[["donor_id", MONTHLY_RELATIONSHIP_ID]].drop_duplicates()
    rel_per_donor = relationships.groupby("donor_id", observed=True).size()

    payment_after_retirement = np.nan
    if MONTHLY_RETIRED_DATE in monthly.columns:
        rel_dates = monthly.groupby(["donor_id", MONTHLY_RELATIONSHIP_ID], observed=True).agg(
            retired=(MONTHLY_RETIRED_DATE, "max"),
            last_payment=(MONTHLY_PAYMENT_DATE, "max"),
        )
        payment_after_retirement = float(
            (rel_dates["retired"].notna()
             & (rel_dates["last_payment"] > rel_dates["retired"])).mean())

    return {
        "rows": len(monthly),
        "donors": n_donors,
        "relationships": len(relationships),
        "donors_multiple_relationships": int((rel_per_donor > 1).sum()),
        "join_date_conflicts": join_conflicts,
        "payment_after_retirement_share": payment_after_retirement,
        "payment_date_min": monthly[MONTHLY_PAYMENT_DATE].min(),
        "payment_date_max": monthly[MONTHLY_PAYMENT_DATE].max(),
        "joined_date_min": monthly[MONTHLY_JOIN_DATE].min(),
        "joined_date_max": monthly[MONTHLY_JOIN_DATE].max(),
    }


def make_monthly_relationships(monthly_pre_T: pd.DataFrame) -> pd.DataFrame:
    """One row per donor + recurring relationship observed through T."""
    cols = ["donor_id", MONTHLY_RELATIONSHIP_ID, "relationship_joined_date",
            "first_payment_date", "last_payment_date"]
    if monthly_pre_T.empty:
        return pd.DataFrame(columns=cols)

    return (monthly_pre_T.groupby(["donor_id", MONTHLY_RELATIONSHIP_ID], observed=True)
            .agg(
                relationship_joined_date=(MONTHLY_JOIN_DATE, "min"),
                first_payment_date=(MONTHLY_PAYMENT_DATE, "min"),
                last_payment_date=(MONTHLY_PAYMENT_DATE, "max"),
            )
            .reset_index())


def prepare(dpr, email, site, monthly, share, project_dates, zip_acs, T, window_months,
            restrict_to_universe: bool = RESTRICT_TO_UNIVERSE,
            pregift_lookback_days: int = PREGIFT_LOOKBACK_DAYS):
    """Normalise inputs and return every slice the feature blocks consume."""
    T = pd.Timestamp(T).normalize()
    W_start = T - pd.DateOffset(months=window_months)
    WP_start = T - pd.DateOffset(months=2 * window_months)
    raw_rows = {"dpr": len(dpr) if dpr is not None else 0}
    plog: list[dict] = []

    dpr = dpr.copy()
    _parse_dates(dpr, "dpr", plog)
    dpr = dpr.loc[dpr["payment_date"].notna()]

    # distance donor -> school, computed once
    if {"donor_lat_long", "school_lat_long"}.issubset(dpr.columns):
        dlat, dlon = _split_latlon(dpr["donor_lat_long"])
        slat, slon = _split_latlon(dpr["school_lat_long"])
        dpr["distance_mi"] = haversine_miles(dlat, dlon, slat, slon)
    else:
        dpr["distance_mi"] = np.nan

    monthly = monthly.copy() if monthly is not None else pd.DataFrame()
    raw_rows["monthly"] = len(monthly)
    _parse_dates(monthly, "monthly", plog)
    monthly_contract = validate_monthly(monthly)

    def cut(df, col, lo=None, hi=None):
        if df.empty or col not in df.columns:
            return df.iloc[0:0]
        m = pd.Series(True, index=df.index)
        if lo is not None:
            m &= df[col] >= lo
        if hi is not None:
            m &= df[col] < hi
        return df.loc[m]

    # ---- phase 1: current behavioral universe ---------------------------------------
    # Project donors qualify on a project gift in W; monthly donors qualify on an actual
    # recurring payment in W. No retirement/status inference is needed to establish activity.
    dpr_W_all = cut(dpr, "payment_date", W_start, T)
    monthly_W_all = cut(monthly, MONTHLY_PAYMENT_DATE, W_start, T)
    ids = [f["donor_id"] for f in (dpr_W_all, monthly_W_all) if not f.empty]
    universe = (pd.Index(pd.unique(pd.concat(ids, ignore_index=True)), name="donor_id")
                if ids else pd.Index([], name="donor_id")).sort_values()

    # ---- phase 2: restrict every source to the universe, then parse the rest ---------
    raw_rows["site"] = len(site) if site is not None else 0
    raw_rows["email"] = len(email) if email is not None else 0
    raw_rows["share"] = len(share) if share is not None else 0

    def to_universe(df):
        if df is None:
            return pd.DataFrame(columns=["donor_id"])
        if restrict_to_universe and not df.empty and "donor_id" in df.columns:
            return df.loc[df["donor_id"].isin(universe)].copy()
        return df.copy()

    dpr = to_universe(dpr)
    monthly = to_universe(monthly)

    site = to_universe(site)
    _parse_dates(site, "site", plog)

    email = to_universe(email)
    if "email_month_start" not in email.columns and "email_sent_month" in email.columns:
        email["email_month_start"] = email["email_sent_month"]
    _parse_dates(email, "email", plog)

    share = to_universe(share)
    if "share_month_start" not in share.columns and "share_sent_month" in share.columns:
        share["share_month_start"] = share["share_sent_month"]
    _parse_dates(share, "share", plog)

    restricted_rows = {"dpr": len(dpr), "monthly": len(monthly), "site": len(site),
                       "email": len(email), "share": len(share)}

    monthly_pre_T = cut(monthly, MONTHLY_PAYMENT_DATE, hi=T)
    monthly_W = cut(monthly, MONTHLY_PAYMENT_DATE, lo=W_start, hi=T)

    S = {
        "T": T, "W_start": W_start, "WP_start": WP_start,
        "dpr_pre_T": cut(dpr, "payment_date", hi=T),
        "dpr_W": cut(dpr, "payment_date", lo=W_start, hi=T),
        "dpr_prevW": cut(dpr, "payment_date", lo=WP_start, hi=W_start),
        "dpr_before_W": cut(dpr, "payment_date", hi=W_start),
        "monthly_pre_T": monthly_pre_T,
        "monthly_W": monthly_W,
        "monthly_relationships": make_monthly_relationships(monthly_pre_T),
        "site_pre_T": cut(site, "activity_date", hi=T),
        "site_W": cut(site, "activity_date", lo=W_start, hi=T),
        "email_pre_T": cut(email, "email_month_start", hi=T),
        "email_W": cut(email, "email_month_start", lo=W_start, hi=T),
        "email_prevW": cut(email, "email_month_start", lo=WP_start, hi=W_start),
        "share_pre_T": cut(share, "share_month_start", hi=T),
        "share_W": cut(share, "share_month_start", lo=W_start, hi=T),
        "project_dates": project_dates,
        "zip_acs": zip_acs,
    }

    # Only site activity within the pre-gift lookback of the window can matter to
    # decision speed, so the cumulative-history build does not need all of pre-T.
    S["site_pregift"] = cut(site, "activity_date",
                            W_start - pd.Timedelta(days=pregift_lookback_days), T)
    S["pregift_lookback_days"] = pregift_lookback_days

    S["restricted_rows"] = restricted_rows
    S["window_months"] = window_months
    S["month_ceiling_daily"] = calendar_months_touched(W_start, T)
    S["month_ceiling_monthly_grain"] = month_starts_in(W_start, T)
    S["parse_log"] = pd.DataFrame(plog, columns=[
        "source", "column", "present", "n_rows", "n_present", "completeness",
        "n_usable", "date_quality", "strategy", "sample_bad", "gates",
        "usable", "verdict"])
    S["input_diagnostics"] = pd.DataFrame([
        {"source": k, "raw_rows": raw_rows.get(k, 0),
         "key_date": kd, "rows_pre_T": len(S[pre]), "w_date": wd, "rows_in_W": len(S[win])}
        for k, pre, win, kd, wd in (
            ("dpr", "dpr_pre_T", "dpr_W", "payment_date", "payment_date"),
            ("email", "email_pre_T", "email_W", "email_month_start", "email_month_start"),
            ("site", "site_pre_T", "site_W", "activity_date", "activity_date"),
            ("monthly", "monthly_pre_T", "monthly_W", MONTHLY_PAYMENT_DATE,
             MONTHLY_PAYMENT_DATE),
            ("share", "share_pre_T", "share_W", "share_month_start", "share_month_start"))])

    pl = S["parse_log"]
    S["date_usable"] = {(r.source, r.column): bool(r.usable)
                        for r in pl.itertuples(index=False)}
    gating = pl.loc[pl["gates"].notna()]
    S["capability_usable"] = {
        (src, cap): bool(raw_rows.get(src, 0) > 0 and not g.empty and g["usable"].all())
        for (src, cap), g in gating.groupby(["source", "gates"], observed=True)}
    S["source_usable"] = {
        src: bool(raw_rows.get(src, 0) > 0
                  and any(k[0] == src for k in S["capability_usable"])
                  and all(v for k, v in S["capability_usable"].items() if k[0] == src))
        for src in ("dpr", "email", "site", "monthly", "share")}

    S["monthly_contract"] = monthly_contract
    S["as_of_extract_columns"] = detect_as_of_extract(S["dpr_pre_T"])
    S["universe"] = universe
    return S

## Section 4 — Block 0/1: population, gating, scaffolding

In [5]:
def blk_population(S) -> pd.DataFrame:
    idx, T, W_start = S["universe"], S["T"], S["W_start"]
    out = pd.DataFrame(index=idx)

    out["snapshot_date"] = T
    out["window_start"] = W_start

    has_proj = pd.Series(idx.isin(S["dpr_W"]["donor_id"].unique()), index=idx).astype("float64")
    has_monthly = pd.Series(
        idx.isin(S["monthly_W"]["donor_id"].unique()), index=idx).astype("float64")

    out["has_project_gift_12m"] = has_proj
    out["has_monthly_payment_12m"] = has_monthly
    out["donor_type_12m"] = np.select(
        [(has_proj == 1) & (has_monthly == 1),
         (has_proj == 1) & (has_monthly == 0),
         (has_proj == 0) & (has_monthly == 1)],
        ["hybrid", "project_only", "monthly_only"], default="inactive")

    # Monthly-only donors are retained in the output but excluded from the clustering fit:
    # project-choice features are structurally absent for them, so joint imputation would
    # manufacture a cluster from missingness. They can be handled as a declared segment.
    out["cluster_eligible"] = has_proj

    first_gift = per_donor(S["dpr_pre_T"], "payment_date", "min", idx)
    fg = pd.to_datetime(first_gift)
    out["first_gift_in_window"] = (fg >= W_start).astype("float64").where(fg.notna())

    out["has_site_data_12m"] = pd.Series(
        idx.isin(S["site_W"]["donor_id"].unique()), index=idx).astype("float64")
    out["has_email_data_12m"] = pd.Series(
        idx.isin(S["email_W"]["donor_id"].unique()), index=idx).astype("float64")
    out["has_share_data_12m"] = pd.Series(
        idx.isin(S["share_W"]["donor_id"].unique()), index=idx).astype("float64")
    return out


def blk_scaffold(S) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)

    n = count_per_donor(dpr_W, idx)
    out["n_gifts_12m"] = n
    out["is_single_gift_12m"] = (n == 1).astype("float64").where(n.notna())

    if dpr_W.empty:
        out["n_active_months_12m"] = np.nan
    else:
        m = dpr_W["payment_date"].dt.to_period("M").astype("string")
        out["n_active_months_12m"] = (m.groupby(dpr_W["donor_id"], observed=True)
                                      .nunique().reindex(idx).astype("float64"))
    out["shares_reliable_12m"] = (n >= RELIABLE_SHARE_MIN_GIFTS).astype("float64").where(n.notna())
    return out

## Section 5 — Block 2: value & volume (PROFILE only)

Built and written, excluded from the clustering matrix by role.

In [6]:
def blk_value(S) -> pd.DataFrame:
    idx = S["universe"]
    dpr_W, dpr_prev, dpr_life = S["dpr_W"], S["dpr_prevW"], S["dpr_pre_T"]
    out = pd.DataFrame(index=idx)

    out["gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "sum", idx)
    out["gift_amount_prev12m"] = per_donor(dpr_prev, "payment_amount", "sum", idx)
    out["n_gifts_prev12m"] = count_per_donor(dpr_prev, idx)
    out["mean_gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "mean", idx)
    out["median_gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "median", idx)
    out["max_gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "max", idx)
    out["min_gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "min", idx)
    out["std_gift_amount_12m"] = per_donor(dpr_W, "payment_amount", "std", idx)

    out["lifetime_amount"] = per_donor(dpr_life, "payment_amount", "sum", idx)
    out["lifetime_gift_count"] = count_per_donor(dpr_life, idx)
    out["lifetime_max_gift_amount"] = per_donor(dpr_life, "payment_amount", "max", idx)
    out["max_donation_sequence_number"] = per_donor(dpr_life, "donation_n", "max", idx)

    # Monthly rows are verified payments, so W dollars are a direct sum. A donor with no
    # monthly payment in W has a real zero; the source contract is validated before feature work.
    md_W = S["monthly_W"]
    out["monthly_amount_12m"] = per_donor(
        md_W, MONTHLY_PAYMENT_AMOUNT, "sum", idx).fillna(0.0)

    # Project and monthly dollars are two observed components of the same 12m total.
    out["grand_amount_12m"] = (out["gift_amount_12m"].fillna(0.0)
                               + out["monthly_amount_12m"])
    denom = out["gift_amount_12m"].fillna(0.0) + out["monthly_amount_12m"]
    out["share_amount_monthly_12m"] = safe_div(out["monthly_amount_12m"], denom)

    out["is_major_gift_donor"] = share_flag(dpr_life, "major_gift_donor", idx).gt(0).astype("float64")
    # Value ON the latest gift row. `groupby.last()` returns the last NON-NULL value, so a
    # balance of 5 on an older gift was reported as current even when the latest gift
    # carried no balance -- the same failure mode already fixed in `blk_latest`.
    out["account_credit_balance_at_latest_gift"] = latest_row_value(
        dpr_life, "account_credit_balance", idx)
    out["max_account_credit_balance"] = per_donor(
        dpr_life, "account_credit_balance", "max", idx)
    out["ever_used_account_credit"] = (
        per_donor(dpr_life, "account_credit_balance", "max", idx).gt(0).astype("float64"))
    return out

## Section 6 — Blocks 3–5: rhythm, velocity, tenure

In [7]:
def blk_rhythm(S) -> pd.DataFrame:
    idx, dpr_W, T = S["universe"], S["dpr_W"], S["T"]
    out = pd.DataFrame(index=idx)

    last = per_donor(S["dpr_pre_T"], "payment_date", "max", idx)
    out["days_since_last_gift"] = (T - pd.to_datetime(last)).dt.days.astype("float64")

    if dpr_W.empty:
        for c in ("days_since_second_to_last_gift", "mean_gap_between_gifts_12m",
                  "cv_gap_between_gifts_12m", "max_gifts_in_calendar_month_12m",
                  "gifts_per_active_month_12m", "is_multi_gift_month_donor_12m",
                  "gift_span_days_12m", "share_gifts_in_burst_7d_12m"):
            out[c] = np.nan
        return out

    d = sort_gifts(dpr_W[["donor_id", "payment_date"]])
    g = d.groupby("donor_id", observed=True)

    # Computed from LIFETIME history so it pairs with `days_since_last_gift`. Restricting
    # it to W returned NaN for a donor whose previous gift sat just outside the window,
    # despite that gift being known.
    life_d = sort_gifts(S["dpr_pre_T"][["donor_id", "payment_date"]])
    desc = life_d.iloc[::-1].copy()
    desc["rk"] = desc.groupby("donor_id", observed=True).cumcount()
    second = desc.loc[desc["rk"] == 1].set_index("donor_id")["payment_date"]
    out["days_since_second_to_last_gift"] = (
        (T - pd.to_datetime(second.reindex(idx))).dt.days.astype("float64"))

    gap = (d["payment_date"] - g["payment_date"].shift(1)).dt.days.astype("float64")
    gap_g = gap.groupby(d["donor_id"], observed=True)
    gap_mean = gap_g.mean()
    out["mean_gap_between_gifts_12m"] = gap_mean.reindex(idx)
    out["cv_gap_between_gifts_12m"] = safe_div(gap_g.std(), gap_mean).reindex(idx)

    span = (g["payment_date"].max() - g["payment_date"].min()).dt.days.astype("float64")
    out["gift_span_days_12m"] = span.reindex(idx)

    ym = dpr_W["payment_date"].dt.to_period("M").astype("string")
    per_month = dpr_W.groupby(["donor_id", ym], observed=True).size()
    mx = per_month.groupby(level=0, observed=True).max()
    out["max_gifts_in_calendar_month_12m"] = mx.reindex(idx).astype("float64")
    out["is_multi_gift_month_donor_12m"] = mx.ge(2).reindex(idx).astype("float64")

    n_gifts = count_per_donor(dpr_W, idx)
    n_months = per_month.groupby(level=0, observed=True).size().reindex(idx).astype("float64")
    out["gifts_per_active_month_12m"] = safe_div(n_gifts, n_months)

    # A burst is a property of consecutive PAIRS, so the denominator is the number of gaps
    # (n-1), not the number of gifts. The first gift in the window has no predecessor and
    # is not evidence of non-bursting.
    out["share_gifts_in_burst_7d_12m"] = share_where(gap <= 7, gap.notna(), d, idx)
    return out


def blk_velocity(S) -> pd.DataFrame:
    idx = S["universe"]
    dpr_W, dpr_prev, before_W = S["dpr_W"], S["dpr_prevW"], S["dpr_before_W"]
    out = pd.DataFrame(index=idx)

    amt_w = per_donor(dpr_W, "payment_amount", "sum", idx).fillna(0.0)
    amt_p = per_donor(dpr_prev, "payment_amount", "sum", idx).fillna(0.0)
    cnt_w = count_per_donor(dpr_W, idx, fill=0.0)
    cnt_p = count_per_donor(dpr_prev, idx, fill=0.0)

    out["amount_velocity_12m_vs_prev12m"] = bounded_ratio(amt_w, amt_p, k=1.0)
    out["count_velocity_12m_vs_prev12m"] = bounded_ratio(cnt_w, cnt_p, k=1.0)

    gave_prev = cnt_p > 0
    gave_before_prev = pd.Series(
        idx.isin(before_W.loc[before_W["payment_date"] < S["WP_start"], "donor_id"].unique())
        if not before_W.empty else False, index=idx)
    gave_w = cnt_w > 0

    out["is_new_donor_12m"] = (gave_w & ~gave_prev & ~gave_before_prev).astype("float64")
    out["is_reactivated_12m"] = (gave_w & ~gave_prev & gave_before_prev).astype("float64")
    out["is_continuing_donor_12m"] = (gave_w & gave_prev).astype("float64")
    return out


def blk_tenure(S) -> pd.DataFrame:
    idx, T = S["universe"], S["T"]
    out = pd.DataFrame(index=idx)
    first = pd.to_datetime(per_donor(S["dpr_pre_T"], "payment_date", "min", idx))
    out["first_donation_date"] = first
    out["tenure_days"] = (T - first).dt.days.astype("float64")
    out["tenure_years"] = out["tenure_days"] / 365.25
    out["first_donation_month"] = first.dt.month.astype("float64")
    out["first_donation_quarter"] = first.dt.quarter.astype("float64")

    # `first_donation_date` is project-only. The any-giving tenure uses the earliest known
    # project gift, recurring payment, or recurring-relationship join observed by T.
    first_pay = per_donor(S["monthly_pre_T"], MONTHLY_PAYMENT_DATE, "min", idx)
    rel = S["monthly_relationships"]
    first_join = per_donor(rel, "relationship_joined_date", "min", idx)
    out["first_monthly_payment_date"] = pd.to_datetime(first_pay)
    earliest = pd.concat([first, pd.to_datetime(first_pay), pd.to_datetime(first_join)],
                         axis=1).min(axis=1)
    out["tenure_days_any_giving"] = (T - earliest).dt.days.astype("float64")
    return out

## Section 7 — Block 6: entry point

Two distinct concepts: the channel of the donor's *first ever* gift (acquisition), and
their channel *mix within W* (ongoing behaviour). The first is a single observation and
unreliable when the extract truncates history; the second is a composition and better
behaved. Both are built.

`referral_channel` returns `<NA>` when both source fields are missing and reserves `"oth"`
for genuinely observed unrecognised values, so the `_unknown` share bucket distinguishes
absent acquisition data from known-but-other traffic.

One-hot level prevalence is measured among **cluster-eligible** donors — those with a gift
inside `W`. An observed entry channel is not sufficient: a donor with historical project
gifts but none in `W` sits in the universe via monthly giving and is excluded from the fit,
so counting them lets a channel that no clustered donor uses outvote ones they do.

In [8]:
VALID_MEDIA = {"email", "directlink", "facebook", "nextdoor", "sharetray",
               "mobilesharetray", "page", "ig", "sendfriend"}
VALID_SOURCES = {"dc", "google"}


def referral_channel(df: pd.DataFrame) -> pd.Series:
    """Collapse referral_medium / referral_source into one channel label.

    `<NA>` when BOTH fields are missing; `"oth"` only for a genuinely observed value that
    is not on the recognised list. Mapping missing to `"oth"` made the `_unknown` share
    bucket unreachable and hid absent acquisition data inside known-but-other traffic.
    """
    if df.empty:
        return pd.Series(dtype="string")
    blank = pd.Series(pd.NA, index=df.index, dtype="string")
    med = (df["referral_medium"].astype("string").str.strip().str.lower()
           if "referral_medium" in df.columns else blank)
    src = (df["referral_source"].astype("string").str.strip().str.lower()
           if "referral_source" in df.columns else blank)
    resolved = pd.Series(
        np.where(med.isin(VALID_MEDIA), med,
                 np.where(src.isin(VALID_SOURCES), src, "oth")),
        index=df.index, dtype="string")
    return resolved.where(med.notna() | src.notna(), pd.NA)


def blk_entry(S, levels: list[str]) -> pd.DataFrame:
    idx = S["universe"]
    life, dpr_W = S["dpr_pre_T"], S["dpr_W"]
    out = pd.DataFrame(index=idx)

    if life.empty:
        out["entry_channel"] = pd.Series(pd.NA, index=idx, dtype="string")
        out["entry_donation_n"] = np.nan
    else:
        first = sort_gifts(life).groupby("donor_id", observed=True).head(1).copy()
        first["_ch"] = referral_channel(first)
        out["entry_channel"] = first.set_index("donor_id")["_ch"].reindex(idx)
        out["entry_donation_n"] = (first.set_index("donor_id")["donation_n"]
                                   .reindex(idx).astype("float64")
                                   if "donation_n" in first.columns else np.nan)

    # One-hots for the k-means path. Level prevalence is measured among donors who will
    # ACTUALLY BE CLUSTERED: an observed entry channel is not sufficient, because a donor
    # with historical project gifts but none inside W is in the universe via monthly
    # giving and has `cluster_eligible = 0`. Measuring across all channel-observed donors
    # let that group outvote the eligible population and select the wrong levels.
    ec = out["entry_channel"]
    observed = ec.notna()
    out["has_entry_channel"] = observed.astype("float64")
    in_fit = observed & pd.Series(idx.isin(dpr_W["donor_id"].unique()), index=idx)
    if in_fit.any():
        keep = pick_levels(pd.Series(ec.index[in_fit], index=ec.index[in_fit]), ec[in_fit])
        for lvl in keep:
            # values populated for every donor with an observed channel (useful for
            # profiling); only the LEVEL SELECTION is restricted to the fit population
            out[f"entry_channel_{slug(lvl)}"] = (ec == lvl).astype("float64").where(observed)

    out["is_teacher"] = share_flag(life, "is_teacher", idx).gt(0).astype("float64")
    out["is_teacher_referred"] = share_flag(life, "is_teacher_referred", idx).gt(0).astype("float64")
    out["is_marketing_subscribed"] = share_flag(
        life, "subscribed_to_marketing_emails", idx).gt(0).astype("float64")

    if not dpr_W.empty:
        w = dpr_W.copy()
        w["_ch"] = referral_channel(w)
        out = out.join(share_matrix(w, "_ch", levels, "share_gifts_channel_", "_12m", idx))
        out["n_unique_channels_12m"] = nunique_where_observed(w["_ch"], w["donor_id"], idx)
        out["entropy_channel_norm_12m"] = group_entropy(w, "_ch", idx, normalize=True)
    return out

## Section 8 — Block 7: recurring monthly giving

The monthly source has one fixed data contract, established from the source itself:

- **one row = one recurring payment**;
- **`recurring_donation_id` = one recurring-giving relationship**.

The build does not infer grain, reconstruct retirement episodes, or guess whether a row is a
pledge versus a charge. `validate_monthly()` fails loudly if the required contract changes.

Three simple objects drive the features:

1. `monthly_pre_T` — payment rows observed before `T`;
2. `monthly_W` — payment rows inside the 12-month window;
3. `monthly_relationships` — one row per donor + recurring relationship observed by `T`.

Payment rows provide dollars, counts, active months, recency, amount variability, and an
observed consecutive-payment-month streak. Relationship IDs provide relationship count and
join timing. `monthly_subscription_retired_date` is QA-only because 4.6% of relationships in
the explored source had payments after their recorded retirement date.

Current monthly activity is **behavioral and snapshot-safe**: a recurring relationship is active
at `T` when its most recent observed payment before `T` is within the prior 30 days. Donor
status is 1 when at least one recurring relationship meets that rule. No extract-time active
flag is used.


In [9]:
def blk_monthly(S) -> pd.DataFrame:
    """Observable recurring-payment and recurring-relationship behavior."""
    idx, T, W_start = S["universe"], S["T"], S["W_start"]
    md, md_W, rel = S["monthly_pre_T"], S["monthly_W"], S["monthly_relationships"]
    out = pd.DataFrame(index=idx)

    # --- 12m payment behavior ----------------------------------------------------
    out["monthly_payments_12m"] = count_per_donor(md_W, idx, fill=0.0)
    if md_W.empty:
        out["monthly_active_months_12m"] = 0.0
        out["monthly_median_payment_amount_12m"] = np.nan
        out["monthly_payment_amount_cv_12m"] = np.nan
    else:
        pay_month = md_W[MONTHLY_PAYMENT_DATE].dt.to_period("M")
        out["monthly_active_months_12m"] = (
            pay_month.groupby(md_W["donor_id"], observed=True).nunique()
            .reindex(idx).fillna(0.0).astype("float64"))
        out["monthly_median_payment_amount_12m"] = per_donor(
            md_W, MONTHLY_PAYMENT_AMOUNT, "median", idx)
        g = md_W.groupby("donor_id", observed=True)[MONTHLY_PAYMENT_AMOUNT]
        out["monthly_payment_amount_cv_12m"] = safe_div(g.std(), g.mean()).reindex(idx)

    # --- payment history through T ----------------------------------------------
    if md.empty:
        for c in ("days_since_last_monthly_payment", "months_since_first_monthly_payment",
                  "monthly_payment_months_lifetime", "monthly_longest_streak_months"):
            out[c] = np.nan
    else:
        g = md.groupby("donor_id", observed=True)[MONTHLY_PAYMENT_DATE]
        first_pay, last_pay = g.min(), g.max()
        out["days_since_last_monthly_payment"] = (
            (T - last_pay).dt.days.astype("float64").reindex(idx))
        out["months_since_first_monthly_payment"] = completed_months(
            pd.Series(T, index=first_pay.index), first_pay).reindex(idx)

        pm = pd.DataFrame({
            "donor_id": md["donor_id"].to_numpy(),
            "m": md[MONTHLY_PAYMENT_DATE].dt.to_period("M").astype("int64").to_numpy(),
        }).drop_duplicates().sort_values(["donor_id", "m"], kind="mergesort")
        out["monthly_payment_months_lifetime"] = (
            pm.groupby("donor_id", observed=True).size().reindex(idx).astype("float64"))
        new_run = pm["donor_id"].ne(pm["donor_id"].shift()) | pm["m"].diff().ne(1)
        pm["run"] = new_run.cumsum()
        runs = pm.groupby(["donor_id", "run"], observed=True).size()
        out["monthly_longest_streak_months"] = (
            runs.groupby(level=0, observed=True).max().reindex(idx).astype("float64"))

    # --- recurring relationships ------------------------------------------------
    if rel.empty:
        out["n_recurring_donation_ids"] = 0.0
        out["months_since_first_monthly_join"] = np.nan
        out["monthly_relationship_started_in_window"] = 0.0
        out["monthly_joined_before_first_project_gift"] = np.nan
    else:
        rg = rel.groupby("donor_id", observed=True)
        out["n_recurring_donation_ids"] = (
            rg.size().reindex(idx).fillna(0.0).astype("float64"))
        first_join = rg["relationship_joined_date"].min()
        out["months_since_first_monthly_join"] = completed_months(
            pd.Series(T, index=first_join.index), first_join).reindex(idx)
        started = rel["relationship_joined_date"].ge(W_start) & rel["relationship_joined_date"].lt(T)
        out["monthly_relationship_started_in_window"] = (
            started.groupby(rel["donor_id"], observed=True).max()
            .reindex(idx).fillna(False).astype("float64"))

        fj = pd.to_datetime(first_join.reindex(idx))
        first_proj = pd.to_datetime(per_donor(S["dpr_pre_T"], "payment_date", "min", idx))
        out["monthly_joined_before_first_project_gift"] = (
            (fj < first_proj).astype("float64").where(fj.notna() & first_proj.notna()))

    # --- current behavioral status ---------------------------------------------
    # Snapshot-safe proxy: a recurring relationship is active when its latest payment
    # observed before T falls within the prior 30 days.
    if rel.empty:
        active_n = pd.Series(0.0, index=idx)
    else:
        cutoff = T - pd.Timedelta(days=MONTHLY_ACTIVE_LOOKBACK_DAYS)
        recent = rel["last_payment_date"].ge(cutoff)
        active_n = (recent.groupby(rel["donor_id"], observed=True).sum()
                    .reindex(idx).fillna(0.0).astype("float64"))
    out["n_active_recurring_donation_ids"] = active_n
    out["is_monthly_donor_current"] = active_n.gt(0).astype("float64")

    return out

## Section 9 — Block 8: timing / seasonality

Seasonal buckets are **mutually exclusive and sum to 1.0**. The original fields
overlapped (Dec 24–31 is a subset of December), which double-counts in a distance
metric. Precedence, first match wins:

1. `year_end_final_week` — Dec 24–31
2. `giving_tuesday_window` — Tuesday after Thanksgiving through the following Sunday
3. `december_other`
4. `back_to_school` — Aug 1 – Sep 30
5. `teacher_appreciation` — May 1–14
6. `summer` — Jun 1 – Jul 31
7. `other`

In [10]:
SEASON_LEVELS = ["year_end_final_week", "giving_tuesday_window", "december_other",
                 "back_to_school", "teacher_appreciation", "summer", "other"]


def giving_tuesday(year: int) -> pd.Timestamp:
    """Tuesday after the fourth Thursday of November."""
    nov1 = pd.Timestamp(year=year, month=11, day=1)
    first_thu = nov1 + pd.Timedelta(days=(3 - nov1.dayofweek) % 7)
    return first_thu + pd.Timedelta(days=21 + 5)


def season_bucket(dates: pd.Series) -> pd.Series:
    d = pd.to_datetime(dates)
    yrs = d.dt.year.dropna().unique()
    gt = {int(y): giving_tuesday(int(y)) for y in yrs}
    gt_start = d.dt.year.map(gt)
    gt_end = gt_start + pd.Timedelta(days=5)
    month, day = d.dt.month, d.dt.day
    conds = [
        (month == 12) & (day >= 24),
        (d >= gt_start) & (d <= gt_end),
        month == 12,
        month.isin([8, 9]),
        (month == 5) & (day <= 14),
        month.isin([6, 7]),
    ]
    return pd.Series(np.select(conds, SEASON_LEVELS[:-1], default="other"),
                     index=d.index, dtype="string")


def blk_timing(S) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)

    if dpr_W.empty:
        return out

    w = dpr_W.copy()
    w["_season"] = season_bucket(w["payment_date"])
    w["_month"] = w["payment_date"].dt.month.astype("Int64").astype("string").str.zfill(2)
    w["_dow"] = w["payment_date"].dt.dayofweek

    out = out.join(share_matrix(w, "_season", SEASON_LEVELS, "share_gifts_", "_12m",
                               idx, fixed_levels=True))
    out = out.join(share_matrix(w, "_season", SEASON_LEVELS, "share_amount_", "_12m",
                               idx, weight="payment_amount", fixed_levels=True))
    out = out.join(share_matrix(w, "_month", [f"{m:02d}" for m in range(1, 13)],
                               "month_share_count_", "_12m", idx, fixed_levels=True))

    out["entropy_gift_month_12m"] = group_entropy(w, "_month", idx)
    out["entropy_gift_month_norm_12m"] = group_entropy(
        w, "_month", idx, normalize=True, n_levels=12)   # a 12m window has 12 months

    cnt = w.groupby(["donor_id", "_month"], observed=True).size()
    out["top_month_share_12m"] = safe_div(
        cnt.groupby(level=0, observed=True).max().astype("float64"),
        cnt.groupby(level=0, observed=True).sum().astype("float64")).reindex(idx)

    out["share_gifts_weekend_12m"] = share_expr(w["_dow"].isin([5, 6]), w, idx)
    out["share_amount_weekend_12m"] = share_flag(
        w.assign(_we=w["_dow"].isin([5, 6]).astype(int)), "_we", idx, weight="payment_amount")
    out["share_gifts_q4_12m"] = share_expr(w["payment_date"].dt.quarter == 4, w, idx)
    return out

## Section 10 — Block 9: gift-size behaviour

Style rather than scale, so these belong in the clustering matrix even though Block 2
does not. `modal_amount_share_12m` is more robust than CV at low gift counts.

Project-cost ratios are computed over gifts with a **known** project cost, with
`coverage_project_cost_12m` reporting how much of the donor's history that was.

In [11]:
ROUND_AMOUNTS = {10, 15, 20, 25, 50, 75, 100, 150, 200, 250, 500, 1000}


def blk_gift_size(S) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out

    w = dpr_W
    amt = w["payment_amount"].astype("float64")
    g = w["donor_id"]

    mean = amt.groupby(g, observed=True).mean()
    out["cv_gift_amount_12m"] = safe_div(amt.groupby(g, observed=True).std(), mean).reindex(idx)

    modal_n = (w.assign(_a=amt.round(2)).groupby(["donor_id", "_a"], observed=True).size()
               .groupby(level=0, observed=True).max().astype("float64"))
    n = count_per_donor(w, idx)
    out["modal_amount_share_12m"] = safe_div(modal_n.reindex(idx), n)
    out["is_fixed_amount_donor_12m"] = (
        (out["modal_amount_share_12m"] >= FIXED_AMOUNT_MODAL_SHARE) & (n >= 3)
    ).astype("float64").where(n.notna())

    is_round = amt.isin(ROUND_AMOUNTS) | ((amt % 25 == 0) & (amt > 0))
    out["share_gifts_round_amount_12m"] = share_where(is_round, amt.notna(), w, idx)

    if "project_total_cost" in w.columns:
        ratio = safe_div(amt, w["project_total_cost"].astype("float64"))
        rg = ratio.groupby(g, observed=True)
        out["mean_gift_to_project_cost_ratio_12m"] = rg.mean().reindex(idx)
        out["median_gift_to_project_cost_ratio_12m"] = rg.median().reindex(idx)
        out["max_gift_to_project_cost_ratio_12m"] = rg.max().reindex(idx)
        # Denominator is gifts with a KNOWN project cost. A gift whose project cost is
        # missing is not evidence of a small contribution, but `NaN >= 0.5` is False.
        out["share_gifts_over_half_project_cost_12m"] = share_where(
            ratio >= 0.5, ratio.notna(), w, idx)
        out["share_gifts_full_project_cost_12m"] = share_where(
            ratio >= 1.0, ratio.notna(), w, idx)
        out["coverage_project_cost_12m"] = coverage(ratio.notna(), w, idx)
        out["median_project_total_cost_12m"] = per_donor(w, "project_total_cost", "median", idx)
        out["mean_project_total_cost_12m"] = per_donor(w, "project_total_cost", "mean", idx)

    if "optional_donation_rate" in w.columns:
        out["avg_optional_donation_rate_12m"] = per_donor(w, "optional_donation_rate", "mean", idx)
        out["share_gifts_with_optional_donation_12m"] = share_expr(
            w["optional_donation_rate"].astype("float64").fillna(0) > 0, w, idx)
    return out

## Section 11 — Block 10: project state at time of gift

Project position is a four-way exclusive composition: a gift can be both first and last
on a project (sole funder), which the original two overlapping flags could not express.

Project lifecycle features use `project_expiration_date` carried directly on each DPR gift.
DonorsChoose projects are open for four calendar months and expiration does not change once
set, so the opening date is inferred as expiration minus four calendar months. Lifecycle
position is scaled from 0 (opening) to 1 (expiration); late-cycle means the final 25% of the
lifecycle, and expiring-soon means 0–7 days remain. Gifts outside the inferred open/expire
interval are treated as unmeasurable rather than clipped.

`gift_completed_project_funding` is different from eventual project outcome: it records
whether this gift itself completed the project's funding at the time of the gift.


In [12]:
POSITION_LEVELS = ["sole_funder", "first_money_in", "closed_project", "mid_funding"]


def blk_project_state(S) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out

    w = dpr_W.copy()
    has_first = "gift_is_projects_first" in w.columns
    has_last = "gift_is_projects_last" in w.columns
    if has_first or has_last:
        f = to_flag(w["gift_is_projects_first"]) if has_first else pd.Series(0.0, index=w.index)
        l = to_flag(w["gift_is_projects_last"]) if has_last else pd.Series(0.0, index=w.index)
        w["_pos"] = np.select(
            [(f == 1) & (l == 1), (f == 1) & (l == 0), (f == 0) & (l == 1)],
            POSITION_LEVELS[:-1], default="mid_funding")
        out = out.join(share_matrix(w, "_pos", POSITION_LEVELS, "share_gifts_", "_12m",
                                   idx, fixed_levels=True))

    if "match_xyi_multiplier" in w.columns:
        mult = w["match_xyi_multiplier"].astype("float64")
        # No `optional_donation_rate` filter: the original restricted to rows where that
        # field was non-null, so donors with matched gifts but no optional-donation rows
        # returned NaN.
        out["share_gifts_with_match_12m"] = share_expr(mult > 1.0, w, idx)
        excess = (mult.fillna(1.0) - 1.0).clip(lower=0.0)
        out["mean_match_excess_12m"] = (excess.groupby(w["donor_id"], observed=True)
                                        .mean().reindex(idx).astype("float64"))
        out["max_match_excess_12m"] = (excess.groupby(w["donor_id"], observed=True)
                                       .max().reindex(idx).astype("float64"))
        out["share_amount_with_match_12m"] = share_flag(
            w.assign(_m=(mult > 1.0).astype(int)), "_m", idx, weight="payment_amount")

    if "gift_completed_project_funding" in w.columns:
        observed = w["gift_completed_project_funding"].notna()
        completed = to_flag(w["gift_completed_project_funding"]) == 1
        out["share_gifts_completing_project_funding_12m"] = share_where(
            completed, observed, w, idx)

    ff = next((c for c in ("project_got_fully_funded", "project_fully_funded")
               if c in w.columns), None)
    if ff is not None:
        # Gift-weighted, not distinct-project-weighted: without `project_id` the same
        # project cannot be deduplicated across a donor's gifts. Named accordingly.
        out["share_gifts_to_fully_funded_projects_12m"] = share_flag(w, ff, idx)
        out["share_gifts_to_never_funded_projects_12m"] = (
            1.0 - out["share_gifts_to_fully_funded_projects_12m"])

    if "is_classroom_essentials_list" in w.columns:
        out["share_gifts_classroom_essentials_12m"] = share_flag(
            w, "is_classroom_essentials_list", idx)
        out["share_amount_classroom_essentials_12m"] = share_flag(
            w, "is_classroom_essentials_list", idx, weight="payment_amount")

    cycle_fields = [
        "funding_cycle_position_mean_12m",
        "share_gifts_late_cycle_12m",
        "days_gift_to_expiration_median_12m",
        "share_gifts_expiring_soon_12m",
        "coverage_project_expiration_12m",
    ]
    if "project_expiration_date" in w.columns:
        pay = w["payment_date"]
        exp = w["project_expiration_date"]
        inferred_open = exp - pd.DateOffset(months=PROJECT_LIFECYCLE_MONTHS)
        lifecycle_days = (exp - inferred_open).dt.days.astype("float64")
        elapsed_days = (pay - inferred_open).dt.days.astype("float64")
        days_to_expiration = (exp - pay).dt.days.astype("float64")

        valid = (pay.notna() & exp.notna() & (lifecycle_days > 0)
                 & (pay >= inferred_open) & (pay <= exp))
        cycle_position = safe_div(elapsed_days, lifecycle_days).where(valid)
        g = w["donor_id"]

        out["funding_cycle_position_mean_12m"] = (
            cycle_position.groupby(g, observed=True).mean().reindex(idx).astype("float64"))
        out["share_gifts_late_cycle_12m"] = share_where(
            cycle_position >= LATE_CYCLE_POSITION, valid, w, idx)
        out["days_gift_to_expiration_median_12m"] = (
            days_to_expiration.where(valid).groupby(g, observed=True)
            .median().reindex(idx).astype("float64"))
        out["share_gifts_expiring_soon_12m"] = share_where(
            days_to_expiration.between(0, EXPIRING_SOON_DAYS, inclusive="both"),
            valid, w, idx)
        out["coverage_project_expiration_12m"] = coverage(valid, w, idx)
    else:
        for c in cycle_fields:
            out[c] = np.nan
    return out

## Section 12 — Block 11: location affinity

Proximity shares are computed among gifts with the **required geography observed**. A gift
with no coordinates is not evidence of giving far away, but `NaN <= 5` is False, so
averaging over all rows silently scored unlocatable gifts as "not local". The companion
`coverage_*` fields report how much of each donor's history was measurable at all.

Same-ZIP needs only `donor_zip` and `school_zip`; same-state and school-state breadth need
the ZIP→state crosswalk. These are separate conditionals so a missing state column cannot
null out the ZIP match. `share_gifts_same_zip3_12m` uses the 3-digit USPS sectional-centre
prefix as a metro-proximity proxy — the closest available stand-in given no CBSA or
district crosswalk.

In [13]:
def zip_state_map(zip_acs: pd.DataFrame) -> pd.Series | None:
    """ZIP5 -> state, if the ACS extract carries a state column."""
    if zip_acs is None or zip_acs.empty:
        return None
    zc = next((c for c in zip_acs.columns if c.upper() in {"ZIP5", "ZIP", "ZIPCODE"}), None)
    sc = next((c for c in zip_acs.columns
               if c.lower() in {"state", "state_abbr", "state_code", "usps_state"}), None)
    if zc is None or sc is None:
        return None
    m = zip_acs[[zc, sc]].dropna().copy()
    m["_z"] = pd.to_numeric(m[zc], errors="coerce")
    m = m.dropna(subset=["_z"]).drop_duplicates("_z")
    return pd.Series(m[sc].astype("string").to_numpy(),
                     index=m["_z"].astype("int64").to_numpy())


def blk_location(S, z2s: pd.Series | None) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out

    w = dpr_W
    dist = w["distance_mi"].astype("float64")
    dg = dist.groupby(w["donor_id"], observed=True)
    out["median_distance_mi_12m"] = dg.median().reindex(idx)
    out["mean_distance_mi_12m"] = dg.mean().reindex(idx)
    out["min_distance_mi_12m"] = dg.min().reindex(idx)
    out["max_distance_mi_12m"] = dg.max().reindex(idx)
    out["cv_distance_mi_12m"] = safe_div(dg.std(), dg.mean()).reindex(idx)

    # Denominator is gifts with COORDINATES, not all gifts: an unlocatable gift is not
    # evidence of giving far away.
    has_dist = dist.notna()
    for thr in (5, 15, 50, 100):
        out[f"share_gifts_within_{thr}mi_12m"] = share_where(dist <= thr, has_dist, w, idx)
    out["share_gifts_over_500mi_12m"] = share_where(dist > 500, has_dist, w, idx)
    out["coverage_distance_12m"] = coverage(has_dist, w, idx)

    if "school_zip" in w.columns:
        out["n_unique_school_zips_12m"] = nunique_where_observed(
            w["school_zip"], w["donor_id"], idx)
        out["entropy_school_zip_norm_12m"] = group_entropy(
            w, "school_zip", idx, normalize=True)
    # `n_unique_schools_12m` lives in the loyalty block (12b), not here.

    # Same-ZIP needs only donor_zip and school_zip. Same-state and school-state breadth
    # need the ZIP->state crosswalk. Keeping them in one conditional made a missing state
    # column silently null out the ZIP match too.
    have_zips = {"donor_zip", "school_zip"}.issubset(w.columns)
    if have_zips:
        dz = pd.to_numeric(w["donor_zip"], errors="coerce")
        sz = pd.to_numeric(w["school_zip"], errors="coerce")
        both = dz.notna() & sz.notna()
        out["share_gifts_same_zip_12m"] = share_where(dz == sz, both, w, idx)
        # 3-digit prefix = USPS sectional centre; a rough metro-proximity proxy, and the
        # closest available stand-in given there is no CBSA or district crosswalk.
        z3d = dz.astype("Int64").astype("string").str.zfill(5).str[:3]
        z3s = sz.astype("Int64").astype("string").str.zfill(5).str[:3]
        out["share_gifts_same_zip3_12m"] = share_where(z3d == z3s, both, w, idx)
        out["coverage_zip_12m"] = coverage(both, w, idx)
    else:
        for c in ("share_gifts_same_zip_12m", "share_gifts_same_zip3_12m",
                  "coverage_zip_12m"):
            out[c] = np.nan

    if z2s is not None and have_zips:
        ds = pd.Series(dz.map(z2s).to_numpy(), index=w.index, dtype="string")
        ss = pd.Series(sz.map(z2s).to_numpy(), index=w.index, dtype="string")
        mapped = ds.notna() & ss.notna()
        out["share_gifts_same_state_12m"] = share_where(ds == ss, mapped, w, idx)
        out["n_unique_school_states_12m"] = nunique_where_observed(
            ss, w["donor_id"], idx)
        out["coverage_state_12m"] = coverage(mapped, w, idx)
    else:
        for c in ("share_gifts_same_state_12m", "n_unique_school_states_12m",
                  "coverage_state_12m"):
            out[c] = np.nan
    return out

## Section 13 — Blocks 12/12b: topic and teacher/school loyalty

Count-share for clustering, dollar-share for profiling — dollar shares are
value-weighted and reintroduce the confound the role split exists to prevent.

Repeat/new relationship detection is per `(donor_id, entity_id)` pair. Testing gifts
against a single global set of historical entity ids answers "has any donor supported
this teacher?" rather than "has *this* donor supported this teacher?", and on data where
popular teachers accumulate many donors it inflates the repeat share toward 1.0 for
nearly everyone.

School-equity affinity shares use only gifts where the relevant school flag is observed;
`coverage_school_equity_flags_12m` records the share of gifts where all three school flags
are available so weak evidence can be gated downstream.


In [14]:
def blk_topic(S, cat_levels, grade_levels) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out
    w = dpr_W

    if "project_category" in w.columns:
        out = out.join(share_matrix(w, "project_category", cat_levels,
                                   "share_count_category_", "_12m", idx))
        out = out.join(share_matrix(w, "project_category", cat_levels,
                                   "share_amount_category_", "_12m", idx,
                                   weight="payment_amount"))
        out["n_unique_categories_12m"] = nunique_where_observed(
            w["project_category"], w["donor_id"], idx)
        out["entropy_category_12m"] = group_entropy(w, "project_category", idx)
        out["entropy_category_norm_12m"] = group_entropy(
            w, "project_category", idx, normalize=True)
        cnt = w.groupby(["donor_id", "project_category"], observed=True).size()
        out["top_category_share_count_12m"] = safe_div(
            cnt.groupby(level=0, observed=True).max().astype("float64"),
            cnt.groupby(level=0, observed=True).sum().astype("float64")).reindex(idx)
        top = (cnt.reset_index(name="n")
               .sort_values(["donor_id", "n"], ascending=[True, False])
               .groupby("donor_id", observed=True).head(1)
               .set_index("donor_id")["project_category"])
        out["top_category_name_12m"] = top.reindex(idx).astype("string")

    if "project_grade" in w.columns:
        out = out.join(share_matrix(w, "project_grade", grade_levels,
                                   "share_count_grade_", "_12m", idx))
        out["n_unique_grades_12m"] = nunique_where_observed(
            w["project_grade"], w["donor_id"], idx)
        out["entropy_grade_norm_12m"] = group_entropy(
            w, "project_grade", idx, normalize=True)

    school_flags = {
        "share_gifts_to_low_income_schools_12m": "school_is_low_income",
        "share_gifts_to_historically_underrepresented_race_schools_12m":
            "school_is_historically_underrepresented_race",
        "share_gifts_to_underserved_rural_schools_12m": "school_is_underserved_rural",
    }
    for new, src in school_flags.items():
        if src in w.columns:
            out[new] = share_where(
                to_flag(w[src]) == 1, w[src].notna(), w, idx)
        else:
            out[new] = np.nan

    pair = ["school_is_low_income", "school_is_historically_underrepresented_race"]
    if all(c in w.columns for c in pair):
        pair_observed = w[pair].notna().all(axis=1)
        out["share_gifts_to_low_income_plus_historically_underrepresented_race_schools_12m"] = (
            share_where(
                (to_flag(w[pair[0]]) == 1) & (to_flag(w[pair[1]]) == 1),
                pair_observed, w, idx))
    else:
        out["share_gifts_to_low_income_plus_historically_underrepresented_race_schools_12m"] = np.nan

    equity_cols = [
        "school_is_low_income",
        "school_is_historically_underrepresented_race",
        "school_is_underserved_rural",
    ]
    out["coverage_school_equity_flags_12m"] = (
        coverage(w[equity_cols].notna().all(axis=1), w, idx)
        if all(c in w.columns for c in equity_cols) else np.nan)
    return out


def blk_loyalty(S) -> pd.DataFrame:
    """Teacher/school concentration within W, plus repeat-vs-new relationships.

    Repeat detection is per `(donor_id, entity_id)` pair. An earlier version built one
    global set of historical entity ids and tested each gift against it, which answers
    "has ANY donor supported this teacher before?" -- on data where popular teachers
    accumulate many donors, that set covers most teachers and the repeat share sits near
    1.0 for nearly everyone.

    Two definitions, both retained because they describe different behaviour:
      `share_gifts_repeat_*_12m`            any prior gift by this donor to this entity,
                                            including a relationship formed inside W
      `share_gifts_repeat_*_pre_window_12m` the relationship predates W specifically
    """
    idx, dpr_W, life, before_W = S["universe"], S["dpr_W"], S["dpr_pre_T"], S["dpr_before_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out
    w = dpr_W

    for ent, col in (("teacher", "teacher_id"), ("school", "school_id")):
        if col not in w.columns:
            continue
        out[f"n_unique_{ent}s_12m"] = nunique_where_observed(w[col], w["donor_id"], idx)
        out[f"entropy_{ent}_norm_12m"] = group_entropy(w, col, idx, normalize=True)
        cnt = w.groupby(["donor_id", col], observed=True).size()
        out[f"top_{ent}_share_count_12m"] = safe_div(
            cnt.groupby(level=0, observed=True).max().astype("float64"),
            cnt.groupby(level=0, observed=True).sum().astype("float64")).reindex(idx)
        amt = w.groupby(["donor_id", col], observed=True)["payment_amount"].sum()
        out[f"top_{ent}_share_amount_12m"] = safe_div(
            amt.groupby(level=0, observed=True).max(),
            amt.groupby(level=0, observed=True).sum()).reindex(idx)
        out[f"gifts_per_{ent}_12m"] = safe_div(
            count_per_donor(w, idx), out[f"n_unique_{ent}s_12m"])

        # --- repeat vs new, per (donor, entity) pair ---------------------------------
        if life.empty or col not in life.columns:
            for c in (f"share_gifts_repeat_{ent}_12m", f"share_gifts_new_{ent}_12m",
                      f"share_gifts_repeat_{ent}_pre_window_12m",
                      f"n_repeat_{ent}_relationships_12m"):
                out[c] = np.nan
            continue

        h = (life[["donor_id", col, "payment_date"]].dropna(subset=[col])
             .astype({col: "string"}))
        # Rank distinct DATES per (donor, entity), not rows: two gifts to the same teacher
        # on the same day are one decision episode, not a return visit, and row order
        # within a date is arbitrary.
        u = (h.drop_duplicates()
             .sort_values(["donor_id", col, "payment_date"], kind="mergesort"))
        u["_prior"] = u.groupby(["donor_id", col], observed=True).cumcount()
        h = h.merge(u, on=["donor_id", col, "payment_date"], how="left")
        hw = h.loc[(h["payment_date"] >= S["W_start"]) & (h["payment_date"] < S["T"])]

        if hw.empty:
            for c in (f"share_gifts_repeat_{ent}_12m", f"share_gifts_new_{ent}_12m",
                      f"n_repeat_{ent}_relationships_12m"):
                out[c] = np.nan
        else:
            rep = (hw["_prior"].gt(0).groupby(hw["donor_id"], observed=True)
                   .mean().reindex(idx).astype("float64"))
            out[f"share_gifts_repeat_{ent}_12m"] = rep
            out[f"share_gifts_new_{ent}_12m"] = 1.0 - rep
            out[f"n_repeat_{ent}_relationships_12m"] = fill_zero_where_observed(
                hw.loc[hw["_prior"] > 0].groupby("donor_id", observed=True)[col].nunique(),
                hw["donor_id"].unique(), idx)

        # relationship predates W: pairwise membership test, not a global id set
        if before_W.empty or col not in before_W.columns:
            out[f"share_gifts_repeat_{ent}_pre_window_12m"] = np.nan
        else:
            prior_pairs = pd.MultiIndex.from_frame(
                before_W[["donor_id", col]].dropna().astype("string").drop_duplicates())
            cur = pd.MultiIndex.from_frame(w[["donor_id", col]].astype("string"))
            is_prior = pd.Series(cur.isin(prior_pairs), index=w.index)
            out[f"share_gifts_repeat_{ent}_pre_window_12m"] = share_where(
                is_prior, w[col].notna(), w, idx)

    for c in ("teacher_lifetime_projects_fully_funded", "teacher_lifetime_donations"):
        if c in w.columns:
            out[f"mean_{c}_12m"] = per_donor(w, c, "mean", idx)
    return out

## Section 14 — Block 13: payment / product channel mix

This is the closest available stand-in for the client's "product affinity" question and
it does **not** cover funds or crisis-motivated giving — see caveats.

In [15]:
PAYMENT_FLAGS = [
    ("daf", "daf_payment"),
    ("gift_card", "gift_card_purchase"),
    ("big_event", "payment_on_big_event"),
    ("green", "is_green_payment"),
    ("anonymous", "donation_is_anonymous"),
]


def blk_payment_mix(S) -> pd.DataFrame:
    idx, dpr_W = S["universe"], S["dpr_W"]
    out = pd.DataFrame(index=idx)
    if dpr_W.empty:
        return out
    w = dpr_W

    for name, col in PAYMENT_FLAGS:
        out[f"share_gifts_{name}_12m"] = share_flag(w, col, idx)
        out[f"share_amount_{name}_12m"] = share_flag(w, col, idx, weight="payment_amount")

    if "green_payment_amount" in w.columns:
        out["share_amount_green_optin_12m"] = safe_div(
            per_donor(w, "green_payment_amount", "sum", idx),
            per_donor(w, "payment_amount", "sum", idx))

    if "payment_type" in w.columns:
        lv = pick_levels(w["donor_id"], w["payment_type"])
        out = out.join(share_matrix(w, "payment_type", lv, "share_gifts_paytype_", "_12m", idx))
    return out

## Section 15 — Block 14: email, site, share engagement

"Active months" counts **distinct months**, not rows. The email and share extracts are
monthly in *date grain* but not unique per donor-month — they appear to carry one row per
send batch — so `groupby().size()` counts batches and can exceed the window length. Sums
(`emails_sent_12m`) are unaffected by grain; only period counts were wrong. This has to be
fixed here: once the output is collapsed to donor level, distinct months are unrecoverable.

Email splits into `email_active_months_12m` (months we sent — exposure, so `PROFILE`) and
`email_months_with_opens_12m` (months they engaged — behaviour, so `CLUSTER`). A donor we
observed who never opened scores **0, not NaN**; only donors absent from the extract are
unknown.

Session-level features are computed on **deduplicated session grain**. Row-weighting made
`rows / active_days` not a session rate at all, and over-weighted device, campaign and
duration for sessions that span several source rows.

Social-sharing counts use the `sharing_` prefix: `share_*` is reserved for proportions in
[0, 1] throughout, which makes that an assertable invariant (QA check 7).

In [16]:
def blk_email(S) -> pd.DataFrame:
    idx, T = S["universe"], S["T"]
    eW, eP, eL = S["email_W"], S["email_prevW"], S["email_pre_T"]
    out = pd.DataFrame(index=idx)
    cols = ("email_sent_count", "email_open_count", "email_click_count")
    if eW.empty or not set(cols).issubset(eW.columns):
        for c in ("emails_sent_12m", "emails_opened_12m", "emails_clicked_12m",
                  "email_open_rate_12m", "email_click_rate_12m",
                  "email_click_to_open_rate_12m", "email_open_rate_prev12m",
                  "email_open_rate_velocity_12m_vs_prev12m",
                  "email_click_rate_velocity_12m_vs_prev12m",
                  "days_since_last_email_sent", "email_active_months_12m",
                  "email_months_with_opens_12m", "email_sends_per_active_month_12m"):
            out[c] = np.nan
        return out

    def rates(df):
        """One groupby pass for all three counters instead of three."""
        if df.empty:
            z = pd.Series(np.nan, index=idx, dtype="float64")
            return z, z, z, z, z, z
        agg = (df.groupby("donor_id", observed=True)[list(cols)].sum()
               .reindex(idx).astype("float64"))
        s, o, c = (agg[x] for x in cols)
        return s, o, c, safe_div(o, s), safe_div(c, s), safe_div(c, o)

    s, o, c, orate, crate, cto = rates(eW)
    out["emails_sent_12m"] = s
    out["emails_opened_12m"] = o
    out["emails_clicked_12m"] = c
    out["email_open_rate_12m"] = orate
    out["email_click_rate_12m"] = crate
    out["email_click_to_open_rate_12m"] = cto

    _, _, _, orate_p, crate_p, _ = rates(eP)
    out["email_open_rate_prev12m"] = orate_p
    out["email_open_rate_velocity_12m_vs_prev12m"] = orate - orate_p
    out["email_click_rate_velocity_12m_vs_prev12m"] = crate - crate_p

    # DISTINCT months, not row count. The email extract is monthly in DATE GRAIN but not
    # unique per donor-month (it appears to carry one row per send batch), so
    # `groupby().size()` counts batches and can exceed the window length.
    sent = pd.to_numeric(eW["email_sent_count"], errors="coerce").fillna(0)
    opened = pd.to_numeric(eW["email_open_count"], errors="coerce").fillna(0)
    seen = eW["donor_id"].unique()
    # 0, not NaN, for a donor we observed in the email extract who simply never opened.
    # Grouping the positive rows and reindexing alone would call that unknown.
    out["email_active_months_12m"] = fill_zero_where_observed(
        eW.loc[sent > 0].groupby("donor_id", observed=True)["email_month_start"].nunique(),
        seen, idx)
    out["email_months_with_opens_12m"] = fill_zero_where_observed(
        eW.loc[opened > 0].groupby("donor_id", observed=True)["email_month_start"].nunique(),
        seen, idx)
    out["email_sends_per_active_month_12m"] = safe_div(s, out["email_active_months_12m"])

    sentL = pd.to_numeric(eL.get("email_sent_count", pd.Series(dtype="float64")),
                          errors="coerce").fillna(0) if not eL.empty else pd.Series(dtype="float64")
    last = per_donor(eL.loc[sentL > 0] if not eL.empty else eL,
                     "email_month_start", "max", idx)
    out["days_since_last_email_sent"] = (T - pd.to_datetime(last)).dt.days.astype("float64")
    return out


SITE_PAGE_COLS = ["project_page_visits_day", "teacher_page_visits_day", "search_visits_day"]

# How to reduce `session_duration_min` when a session spans multiple source rows. "max" is
# right if rows repeat or accumulate the session total; switch to "sum" if the source
# splits one session's duration across rows.
SESSION_DURATION_AGG = "max"


def visit_frame(sW: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
    """Collapse site rows to one row per VISIT, and report which grain a visit is.

    A visit is a session when `session_id` is present, otherwise a donor-day. Naming these
    features "session_*" unconditionally is a lie on a donor-day extract with no session
    column; hard-coding "day" would silently discard real session structure if the column
    ever appears. `site_unit_is_session` records which happened, so the meaning of every
    visit-basis field stays recoverable downstream.
    """
    if sW.empty:
        return sW, False
    is_session = "session_id" in sW.columns
    unit = ["donor_id", "session_id"] if is_session else ["donor_id", "activity_date"]
    s = sW.copy()
    if "came_from_campaign" in s.columns:
        s["came_from_campaign"] = to_flag(s["came_from_campaign"])
    s = s.sort_values(unit, kind="mergesort")
    spec = {c: (c, how) for c, how in
            (("activity_date", "first"), ("device_type", "first"),
             ("session_duration_min", SESSION_DURATION_AGG),
             ("came_from_campaign", "max"))
            if c in s.columns and c not in unit}
    if not spec:
        return s.drop_duplicates(unit), is_session
    return s.groupby(unit, observed=True).agg(**spec).reset_index(), is_session


def blk_site(S) -> pd.DataFrame:
    idx, T = S["universe"], S["T"]
    sW, sL = S["site_W"], S["site_pre_T"]
    out = pd.DataFrame(index=idx)
    if sW.empty:
        return out

    visits, is_session = visit_frame(sW)
    out["site_unit_is_session"] = float(is_session)

    days = (sW.groupby("donor_id", observed=True)["activity_date"]
            .nunique().reindex(idx).astype("float64"))
    rows = count_per_donor(sW, idx)
    out["days_with_site_activity_12m"] = days
    out["site_rows_12m"] = rows
    out["site_rows_per_active_day_12m"] = safe_div(rows, days)

    n_visits = count_per_donor(visits, idx)
    out["n_site_visits_12m"] = n_visits
    # Exactly 1.0 when a visit IS a day. `degenerate_fields()` then reports it as constant,
    # which is the honest signal -- better than a field whose name implies session structure
    # the source does not carry.
    out["site_visits_per_active_day_12m"] = safe_div(n_visits, days)

    if "session_duration_min" in visits.columns:
        out["avg_visit_duration_min_12m"] = per_donor(visits, "session_duration_min",
                                                      "mean", idx)
        out["total_visit_duration_min_12m"] = (
            visits.groupby("donor_id", observed=True)["session_duration_min"]
            .sum(min_count=1).reindex(idx).astype("float64"))
    if "came_from_campaign" in visits.columns:
        out["campaign_visit_share_12m"] = share_flag(visits, "came_from_campaign", idx)

    present = [c for c in SITE_PAGE_COLS if c in sW.columns]
    if present:
        # page counts are per-row tallies, so summing is correct at either grain
        sums = sW.groupby("donor_id", observed=True)[present].sum()
        tot = sums.sum(axis=1)
        for c in present:
            out[f"share_{c.replace('_visits_day', '')}_visits_12m"] = safe_div(
                sums[c].astype("float64"), tot.astype("float64")).reindex(idx)
            out[f"{c}_total_12m"] = sums[c].reindex(idx).astype("float64")
        out["page_visits_per_active_day_12m"] = safe_div(tot.reindex(idx).astype("float64"), days)

    if "cart_visits_day" in sW.columns:
        cart = per_donor(sW, "cart_visits_day", "sum", idx)
        out["cart_visits_12m"] = cart
        out["cart_visits_per_active_day_12m"] = safe_div(cart, days)

    if "device_type" in visits.columns:
        dev = visits.assign(_d=visits["device_type"].astype("string").str.strip().str.lower())
        out = out.join(share_matrix(dev, "_d", ["mobile", "desktop", "tablet"],
                                   "device_share_", "_12m", idx))

    if "cart_visits_day" in sL.columns and not sL.empty:
        hc = sL.loc[sL["cart_visits_day"].astype("float64") > 0]
        last_cart = per_donor(hc, "activity_date", "max", idx)
        out["days_since_last_cart_visit"] = (
            (T - pd.to_datetime(last_cart)).dt.days.astype("float64"))
    return out


def blk_share_events(S) -> pd.DataFrame:
    idx, shW = S["universe"], S["share_W"]
    out = pd.DataFrame(index=idx)
    # `sharing_` prefix, not `share_`: everywhere else in this notebook `share_*` denotes a
    # proportion in [0, 1], and these are counts of social-sharing events. The collision
    # made an otherwise clean invariant unassertable.
    if shW.empty or "share_event_count" not in shW.columns:
        for c in ("sharing_events_12m", "sharing_active_months_12m", "is_sharer_12m",
                  "sharing_events_per_active_month_12m"):
            out[c] = np.nan
        return out
    cnt = pd.to_numeric(shW["share_event_count"], errors="coerce")
    observed_ids = shW.loc[cnt.notna(), "donor_id"].unique()
    ev = (cnt.groupby(shW["donor_id"], observed=True).sum(min_count=1)
          .reindex(idx).astype("float64"))
    out["sharing_events_12m"] = ev
    # DISTINCT months among rows with actual activity -- not a row count. Donors with an
    # observed zero count get 0; donors with no usable share count remain NaN.
    out["sharing_active_months_12m"] = fill_zero_where_observed(
        shW.loc[cnt > 0].groupby("donor_id", observed=True)["share_month_start"].nunique(),
        observed_ids, idx)
    out["is_sharer_12m"] = ev.gt(0).astype("float64").where(ev.notna())
    out["sharing_events_per_active_month_12m"] = safe_div(
        ev, out["sharing_active_months_12m"])
    return out

## Section 16 — Block 15: decision speed

Joins site events to gift dates — the one join the original pipeline never made. Uses
`merge_asof` against per-donor cumulative sums so the pre-gift window is computed
without materialising a gift x site-day cross product.

Structural ceiling: site data is donor-day aggregated, so "viewed three projects" and
"reloaded one project three times" are indistinguishable, and there is no within-session
ordering. Check `decision_speed_measurable_12m` before treating this as a real axis.

In [17]:
def blk_decision_speed(S, lookback_days: int | None = None) -> pd.DataFrame:
    # `site_pregift` holds only the rows that can fall inside a pre-gift window, rather than
    # all of pre-T history, so the cumulative build stays proportional to the window.
    lookback_days = lookback_days or S.get("pregift_lookback_days", PREGIFT_LOOKBACK_DAYS)
    idx, dpr_W, site = S["universe"], S["dpr_W"], S["site_pregift"]
    metrics = [c for c in SITE_PAGE_COLS + ["cart_visits_day"] if c in site.columns]
    names = ([f"{m.replace('_visits_day', '')}_pre_gift_mean_12m" for m in metrics]
             + ["site_days_pre_gift_mean_12m", "share_gifts_with_prior_search_12m",
                "share_gifts_same_day_first_session_12m",
                "days_first_session_to_gift_median_12m", "decision_speed_measurable_12m"])
    if dpr_W.empty or site.empty:
        return empty_frame(idx, names)

    # one row per donor-day, metrics summed, plus an activity indicator
    day = (site.groupby(["donor_id", "activity_date"], observed=True)[metrics].sum()
           .reset_index() if metrics
           else site[["donor_id", "activity_date"]].drop_duplicates())
    day["_active"] = 1.0
    day = day.sort_values(["donor_id", "activity_date"], kind="mergesort")
    cum_cols = metrics + ["_active"]
    for c in cum_cols:
        day[f"cum_{c}"] = day.groupby("donor_id", observed=True)[c].cumsum()
    cum = day[["donor_id", "activity_date"] + [f"cum_{c}" for c in cum_cols]]

    gifts = dpr_W[["donor_id", "payment_date"]].copy().reset_index(drop=True)
    gifts["_hi"] = gifts["payment_date"]                                     # inclusive
    gifts["_lo_excl"] = gifts["payment_date"] - pd.Timedelta(days=lookback_days)
    gifts["_lo"] = gifts["payment_date"] - pd.Timedelta(days=lookback_days - 1)

    def asof(left_key: str, direction: str, suffix: str) -> pd.DataFrame:
        """merge_asof requires globally sorted keys, so results come back in sorted
        order -- reindex to gift order before any element-wise comparison."""
        L = gifts.sort_values(left_key, kind="mergesort")
        R = cum.sort_values("activity_date", kind="mergesort")
        m = pd.merge_asof(L, R, left_on=left_key, right_on="activity_date",
                          by="donor_id", direction=direction)
        return m.set_index(L.index).add_suffix(suffix).reindex(gifts.index)

    hi = asof("_hi", "backward", "_hi")
    lo = asof("_lo_excl", "backward", "_lo")

    res = pd.DataFrame(index=gifts.index)
    for c in cum_cols:
        a = hi[f"cum_{c}_hi"].astype("float64").fillna(0.0)
        b = lo[f"cum_{c}_lo"].astype("float64").fillna(0.0)
        res[c] = (a - b).clip(lower=0.0)

    # earliest active day inside the window, via a forward as-of from the window start
    fwd = asof("_lo", "forward", "_fw")
    first_act = pd.to_datetime(fwd["activity_date_fw"])
    in_win = first_act.notna() & (first_act <= gifts["_hi"]) & (first_act >= gifts["_lo"])
    res["_first_act"] = first_act.where(in_win)
    res["_days_to_gift"] = (gifts["_hi"] - res["_first_act"]).dt.days.astype("float64")
    res["donor_id"] = gifts["donor_id"].to_numpy()

    out = pd.DataFrame(index=idx)
    rg = res.groupby("donor_id", observed=True)
    for m in metrics:
        out[f"{m.replace('_visits_day', '')}_pre_gift_mean_12m"] = (
            rg[m].mean().reindex(idx).astype("float64"))
    out["site_days_pre_gift_mean_12m"] = rg["_active"].mean().reindex(idx).astype("float64")

    # The cumulative subtraction needs missing as-of baselines to behave like zero, but a
    # donor with no site rows in W is unobserved, not a confirmed zero-browser. Mirror
    # `has_site_data_12m` here so missing site coverage cannot manufacture a segment.
    pre_gift_mean_cols = ([f"{m.replace('_visits_day', '')}_pre_gift_mean_12m" for m in metrics]
                          + ["site_days_pre_gift_mean_12m"])
    has_site_data = idx.isin(S["site_W"]["donor_id"].unique())
    out.loc[~has_site_data, pre_gift_mean_cols] = np.nan

    # Denominator is gifts with ANY pre-gift site activity. A gift made by a donor we
    # never saw on site is not evidence of a fast decision.
    seen = res["_active"] > 0
    if "search_visits_day" in metrics:
        out["share_gifts_with_prior_search_12m"] = share_where(
            res["search_visits_day"] > 0, seen, res, idx)
    else:
        out["share_gifts_with_prior_search_12m"] = np.nan

    out["share_gifts_same_day_first_session_12m"] = share_where(
        res["_days_to_gift"] == 0, seen, res, idx)
    out["days_first_session_to_gift_median_12m"] = (
        rg["_days_to_gift"].median().reindex(idx).astype("float64"))
    out["decision_speed_measurable_12m"] = (
        res["_active"].gt(0).groupby(res["donor_id"], observed=True)
        .mean().reindex(idx).astype("float64"))
    return out

## Section 17 — Blocks 16/18: latest gift and ZIP demographics

`donor_zip5` is the most recent known project-gift ZIP. The verified monthly payment
source does not contain donor ZIP, so monthly-only donors have no ZIP/ACS profile here rather
than receiving an inferred address.

`.tail(1)` rather than `groupby.last()`: `last()` returns the last **non-null** value per
column, so a null on the most recent gift silently pulls that field from an earlier one.

ZIP/ACS demographics are an addition beyond the spec, defaulted to `PROFILE`. They are a
legitimate segmentation axis but none of the client's nine questions is demographic, and
including them would add a "rich ZIP vs poor ZIP" dimension that is not in the brief.
Flipping them to `CLUSTER` is a one-line change in `ROLE_RULES`.

In [18]:
def blk_latest(S) -> pd.DataFrame:
    idx, life, shL = S["universe"], S["dpr_pre_T"], S["share_pre_T"]
    out = pd.DataFrame(index=idx)
    if life.empty:
        return out

    # `sort_gifts` breaks payment_date ties with donation_n, so two gifts on the same day
    # no longer resolve by source row order.
    latest = (sort_gifts(life).groupby("donor_id", observed=True)
              .tail(1).set_index("donor_id"))

    num = {"latest_gift_amount": "payment_amount",
           "latest_project_total_cost": "project_total_cost",
           "latest_optional_donation_rate": "optional_donation_rate",
           "latest_match_xyi_multiplier": "match_xyi_multiplier",
           "latest_distance_mi": "distance_mi",
           "latest_teacher_lifetime_projects_fully_funded":
               "teacher_lifetime_projects_fully_funded"}
    for new, src in num.items():
        out[new] = (latest[src].reindex(idx).astype("float64")
                    if src in latest.columns else np.nan)

    flags = {"latest_is_green": "is_green_payment",
             "latest_is_giftcard_purchase": "gift_card_purchase",
             "latest_is_anonymous": "donation_is_anonymous",
             "latest_project_got_fully_funded": "project_got_fully_funded",
             "latest_is_projects_first": "gift_is_projects_first",
             "latest_is_projects_last": "gift_is_projects_last",
             "latest_is_big_event": "payment_on_big_event"}
    for new, src in flags.items():
        out[new] = (to_flag(latest[src]).reindex(idx).astype("float64")
                    if src in latest.columns else np.nan)

    for new, src in (("latest_project_category", "project_category"),
                     ("latest_project_grade", "project_grade")):
        out[new] = (latest[src].reindex(idx).astype("string")
                    if src in latest.columns else pd.Series(pd.NA, index=idx, dtype="string"))

    lt = latest.reset_index()
    out["latest_referral_channel"] = (
        pd.Series(referral_channel(lt).to_numpy(), index=lt["donor_id"])
        .reindex(idx).astype("string"))

    out["latest_gift_month"] = (latest["payment_date"].dt.month
                                .reindex(idx).astype("float64"))
    out["latest_gift_season"] = (season_bucket(latest["payment_date"])
                                 .reindex(idx).astype("string"))

    if not shL.empty and {"share_month_start", "share_event_count"}.issubset(shL.columns):
        cnt = pd.to_numeric(shL["share_event_count"], errors="coerce")
        observed = shL.loc[cnt.notna()]
        active = shL.loc[cnt > 0]
        gm = latest["payment_date"].dt.to_period("M").astype("string")
        observed_months = set(zip(
            observed["donor_id"].astype("string"),
            observed["share_month_start"].dt.to_period("M").astype("string")))
        active_months = set(zip(
            active["donor_id"].astype("string"),
            active["share_month_start"].dt.to_period("M").astype("string")))
        keys = list(zip(latest.index.astype("string"), gm))
        out["latest_shared_same_month"] = pd.Series(
            [1.0 if k in active_months else 0.0 if k in observed_months else np.nan
             for k in keys], index=latest.index, dtype="float64").reindex(idx)
    else:
        out["latest_shared_same_month"] = np.nan
    return out


ACS_FIELDS = {
    "zip_pct_households_with_children": "pct_households_with_children",
    "zip_pct_in_labor_force": "pct_in_labor_force",
    "zip_unemployment_rate": "unemployment_rate",
    "zip_pct_single_parent": "pct_single_parent",
    "zip_pct_minority": "pct_minority",
    "zip_avg_household_size": "avg_household_size",
    "zip_median_age": "median_age",
    "zip_median_home_value": "median_home_value",
}


def blk_acs(S) -> pd.DataFrame:
    idx, life = S["universe"], S["dpr_pre_T"]
    acs = S["zip_acs"]
    out = pd.DataFrame(index=idx)

    # Current monthly payment source has no donor ZIP, so the latest known project-gift ZIP
    # is the only point-in-time address available here. Monthly-only donors remain NaN.
    if not life.empty and "donor_zip" in life.columns:
        latest_zip = (sort_gifts(life.loc[life["donor_zip"].notna(),
                                           ["donor_id", "payment_date", "donor_zip"]])
                      .groupby("donor_id", observed=True).tail(1)
                      .set_index("donor_id")["donor_zip"])
        z = pd.to_numeric(latest_zip.reindex(idx), errors="coerce")
    else:
        z = pd.Series(np.nan, index=idx, dtype="float64")
    out["donor_zip5"] = z

    if acs is None or acs.empty:
        for k in ACS_FIELDS:
            out[k] = np.nan
        out["zip_log_total_population"] = np.nan
        return out

    a = acs.copy()
    zc = next((c for c in a.columns if c.upper() in {"ZIP5", "ZIP", "ZIPCODE"}), None)
    if zc is None:
        for k in ACS_FIELDS:
            out[k] = np.nan
        out["zip_log_total_population"] = np.nan
        return out
    a["_z"] = pd.to_numeric(a[zc], errors="coerce")
    a = a.dropna(subset=["_z"]).drop_duplicates("_z").set_index("_z")

    for new, src in ACS_FIELDS.items():
        out[new] = (pd.Series(z.map(a[src]).to_numpy(), index=idx).astype("float64")
                    if src in a.columns else np.nan)
    if "total_population" in a.columns:
        out["zip_log_total_population"] = np.log1p(
            pd.Series(z.map(a["total_population"]).to_numpy(), index=idx).astype("float64"))
    else:
        out["zip_log_total_population"] = np.nan
    return out

## Section 18 — Role assignment

Ordered pattern rules rather than a hand-maintained 200-row dict. Any field matching no
rule falls through to `CLUSTER` and is reported by the QA cell, so a new feature can
never silently acquire the wrong role.

In [19]:
# No feature fields are currently blocked. Keep the set so status/QA logic remains stable
# if a future source limitation requires an explicit blocked field again.
BLOCKED_FIELDS = set()

# Suspected as-of-extract, confirmed at run time by `detect_as_of_extract`: if a teacher's
# lifetime counters never vary across gifts made at different times, they are a single
# current value stamped onto every row rather than the value as of each gift.
AS_OF_EXTRACT_SUSPECTS = {
    "mean_teacher_lifetime_projects_fully_funded_12m": "teacher_lifetime_projects_fully_funded",
    "mean_teacher_lifetime_donations_12m": "teacher_lifetime_donations",
    "latest_teacher_lifetime_projects_fully_funded": "teacher_lifetime_projects_fully_funded",
}


def detect_as_of_extract(dpr: pd.DataFrame, entity: str = "teacher_id") -> dict[str, bool]:
    """Which teacher-lifetime columns are stamped at extract time rather than per gift.

    A per-gift counter grows as the teacher receives more gifts, so it varies across a
    teacher's gifts. A single current value repeated on every row does not. Detecting this
    beats assuming it, and it cannot be recomputed from a donor-cohort extract -- the gifts
    of other donors to the same teacher are not present.
    """
    out: dict[str, bool] = {}
    cols = [c for c in set(AS_OF_EXTRACT_SUSPECTS.values()) if c in dpr.columns]
    if dpr.empty or entity not in dpr.columns or not cols:
        return out
    multi = dpr.groupby(entity, observed=True).size()
    many = multi.index[multi > 1]
    if len(many) == 0:
        return out
    sub = dpr.loc[dpr[entity].isin(many), [entity] + cols]
    for c in cols:
        varies = sub.groupby(entity, observed=True)[c].nunique().max()
        out[c] = bool(pd.isna(varies) or varies <= 1)   # never varies -> as-of-extract
    return out


ROLE_RULES: list[tuple[str, str]] = [
    (r"^donor_id$|^snapshot_date$|^window_start$|^first_donation_date$"
     r"|^first_monthly_payment_date$", "KEY"),
    (r"^(has_|coverage_|cluster_eligible$|shares_reliable_|donor_type_"
     r"|first_gift_in_window$|decision_speed_measurable_|top_category_name_"
     r"|entry_donation_n$|site_unit_is_session$)", "DIAG"),
    # CLUSTER_CAT: string-valued and usable directly by Gower/k-medoids. Each has
    # `entry_channel_<level>` one-hot counterparts in CLUSTER for the k-means path --
    # use one or the other, never both.
    (r"^entry_channel$", "CLUSTER_CAT"),
    (r"^share_amount_", "PROFILE"),
    (r"^month_share_count_", "PROFILE"),
    (r"^latest_", "PROFILE"),
    (r"^lifetime_", "PROFILE"),
    (r"^zip_|^donor_zip5$", "PROFILE"),
    # Sending cadence and raw dollar/count intensity profile the donor; behavioral shape
    # features such as streak, recency, active months and amount CV remain CLUSTER.
    (r"^(gift_amount|mean_gift_amount|median_gift_amount|max_gift_amount|min_gift_amount"
     r"|std_gift_amount|n_gifts|grand_amount|monthly_amount|monthly_payments_12m"
     r"|monthly_median_payment_amount_12m"
     r"|max_donation_sequence_number|is_major_gift_donor|account_credit_balance_at"
     r"|max_account_credit_balance|ever_used_account_credit|tenure_years"
     r"|emails_sent|emails_opened|emails_clicked|email_active_months_12m"
     r"|email_sends_per_active_month_12m|sharing_events_12m|site_rows|n_site_visits"
     r"|total_visit_duration_min|project_page_visits_day_total"
     r"|teacher_page_visits_day_total|search_visits_day_total|cart_visits_12m)", "PROFILE"),
    (r"", "CLUSTER"),
]


def assign_roles(cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in cols:
        role = next(r for pat, r in ROLE_RULES if re.search(pat, c))
        rows.append({"field": c, "role": role})
    return pd.DataFrame(rows)

## Section 19 — Orchestrator

In [20]:
BLOCK_ORDER = [
    ("00_population", lambda S, C: blk_population(S)),
    ("01_scaffold", lambda S, C: blk_scaffold(S)),
    ("02_value", lambda S, C: blk_value(S)),
    ("03_rhythm", lambda S, C: blk_rhythm(S)),
    ("04_velocity", lambda S, C: blk_velocity(S)),
    ("05_tenure", lambda S, C: blk_tenure(S)),
    ("06_entry", lambda S, C: blk_entry(S, C["channel_levels"])),
    ("07_monthly", lambda S, C: blk_monthly(S)),
    ("08_timing", lambda S, C: blk_timing(S)),
    ("09_gift_size", lambda S, C: blk_gift_size(S)),
    ("10_project_state", lambda S, C: blk_project_state(S)),
    ("11_location", lambda S, C: blk_location(S, C["zip_state"])),
    ("12_topic", lambda S, C: blk_topic(S, C["cat_levels"], C["grade_levels"])),
    ("12b_loyalty", lambda S, C: blk_loyalty(S)),
    ("13_payment_mix", lambda S, C: blk_payment_mix(S)),
    ("14a_email", lambda S, C: blk_email(S)),
    ("14b_site", lambda S, C: blk_site(S)),
    ("14c_share", lambda S, C: blk_share_events(S)),
    ("15_decision_speed", lambda S, C: blk_decision_speed(S)),
    ("16_latest", lambda S, C: blk_latest(S)),
    ("18_acs", lambda S, C: blk_acs(S)),
]


# A uniqueness or episode-count verdict drawn from a handful of surviving rows is worse
# than no verdict at all, so the report refuses to conclude below this many rows.
MIN_ROWS_FOR_INVARIANT = 1_000
# Retention below this fraction almost always means a parsing or extract problem, not that
# the donors genuinely have no history in range.
LOW_RETENTION_WARN = 0.05


def report_source_grain(S) -> None:
    """Print source retention, date quality, and the verified monthly contract."""
    print("\nDate columns  (pop = share of rows populated; ok = share of those that are "
          "plausible dates)")
    for r in S["parse_log"].itertuples(index=False):
        gate = f" [gates {r.gates}]" if pd.notna(r.gates) else ""
        if not r.present:
            print(f"  {r.source:<8s} {r.column:<38s} ABSENT from the extract{gate}")
            continue
        if r.verdict == "ALL BLANK":
            print(f"  {r.source:<8s} {r.column:<38s} pop   0.0% (0/{r.n_rows:,}) | "
                  f"all blank, expected for this column{gate}")
            continue
        line = (f"  {r.source:<8s} {r.column:<38s} pop {r.completeness:6.1%} "
                f"({r.n_present:,}/{r.n_rows:,}) | ok {r.date_quality:6.1%} | "
                f"{r.strategy}{gate}")
        if r.verdict == "OK":
            note = ("  <-- RECOVERED as Excel serial dates" if r.strategy == "excel serial"
                    else f"  (dropped: {r.sample_bad})" if r.sample_bad else "")
        elif r.verdict in ("EMPTY", "NEARLY EMPTY"):
            note = f"  <-- {r.verdict}: the column carries too little usable data"
        else:
            note = f"  <-- UNPARSEABLE; raw values look like: {r.sample_bad}"
        print(line + note)

    print("\nInput retention  (kept = rows for donors in the universe):")
    rr = S["restricted_rows"]
    for r in S["input_diagnostics"].itertuples(index=False):
        kept = rr.get(r.source, r.raw_rows)
        cut_pct = (1 - kept / r.raw_rows) if r.raw_rows else 0.0
        pre_ratio = (r.rows_pre_T / kept) if kept else 0.0
        w_ratio = (r.rows_in_W / kept) if kept else 0.0
        collapsed = [n for n, v in ((r.key_date, pre_ratio), (r.w_date, w_ratio))
                     if kept and v < LOW_RETENTION_WARN]
        flag = ("  <-- CHECK: almost nothing survived the date filter on "
                + ", ".join(dict.fromkeys(collapsed))) if collapsed else ""
        print(f"  {r.source:<8s} {r.raw_rows:>12,} raw -> {kept:>12,} kept "
              f"({cut_pct:5.1%} dropped) | {r.rows_pre_T:>12,} pre-T by {r.key_date} "
              f"| {r.rows_in_W:>12,} in W by {r.w_date}{flag}")
        if r.raw_rows in (1_048_575, 1_048_576, 65_535, 65_536):
            print(f"           ^ {r.raw_rows:,} rows is a spreadsheet row limit -- check "
                  "for truncation")

    unusable = [k for k, ok in S["source_usable"].items()
                if not ok and S["input_diagnostics"].set_index("source").loc[k, "raw_rows"] > 0]
    if unusable:
        print(f"\nUNUSABLE SOURCE(S): {', '.join(unusable)}")
        print("Fix those extracts before interpreting their downstream features.")

    q = S["monthly_contract"]
    print("\nMonthly contract: PASS  (row=payment; recurring_donation_id=relationship)")
    print(f"  {q['rows']:,} rows | {q['donors']:,} donors | "
          f"{q['relationships']:,} recurring relationships")
    print(f"  donors with >1 relationship: {q['donors_multiple_relationships']:,}")
    print(f"  payment dates: {q['payment_date_min']:%Y-%m-%d} -> "
          f"{q['payment_date_max']:%Y-%m-%d}")
    print(f"  joined dates:  {q['joined_date_min']:%Y-%m-%d} -> "
          f"{q['joined_date_max']:%Y-%m-%d}")
    print(f"  join-date conflicts within relationship: {q['join_date_conflicts']:,}")
    if pd.notna(q['payment_after_retirement_share']):
        print(f"  payments after recorded retirement: "
              f"{q['payment_after_retirement_share']:.2%} of all relationships "
              "(retired_date is QA-only)")
    print(f"  behavioral active definition: latest payment within "
          f"{MONTHLY_ACTIVE_LOOKBACK_DAYS} days before T")

    aoe = S.get("as_of_extract_columns", {})
    if aoe:
        print("\nAs-of-extract detection (teacher lifetime fields):")
        for col, is_extract in sorted(aoe.items()):
            print(f"  {col:<44s} "
                  + ("AS-OF-EXTRACT -- marked snapshot-unsafe"
                     if is_extract else "varies per gift -> snapshot-safe"))

    print("\nSource grain checks:")
    checks = [
        ("email", S["email_pre_T"], ["donor_id", "email_month_start"]),
        ("share", S["share_pre_T"], ["donor_id", "share_month_start"]),
        ("site", S["site_pre_T"], ["donor_id", "activity_date"]),
        ("monthly", S["monthly_pre_T"],
         ["donor_id", MONTHLY_RELATIONSHIP_ID, MONTHLY_PAYMENT_DATE]),
    ]
    for name, df, keys in checks:
        if df.empty or not set(keys).issubset(df.columns):
            print(f"  {name:<8s} (empty or missing keys -- no verdict)")
            continue
        n, u = len(df), len(df.drop_duplicates(keys))
        if n < MIN_ROWS_FOR_INVARIANT:
            print(f"  {name:<8s} {n:>9,} rows | too few rows to conclude")
            continue
        print(f"  {name:<8s} {n:>9,} rows | {u:>9,} unique {'+'.join(keys)}"
              f" | unique={n == u}" + ("" if n == u else f"  <-- {n / u:.2f} rows/key"))


def build_clustering_features(dpr, email, site, monthly, share, project_dates, zip_acs,
                              T=T, window_months=WINDOW_MONTHS, verbose=True):
    """Return (features, field_meta). No imputation, scaling, or encoding applied."""
    S = prepare(dpr, email, site, monthly, share, project_dates, zip_acs, T, window_months)
    if verbose:
        print(f"T={S['T']:%Y-%m-%d}  W=[{S['W_start']:%Y-%m-%d}, {S['T']:%Y-%m-%d})")
        print(f"universe: {len(S['universe']):,} donors | "
              f"dpr_W {len(S['dpr_W']):,} rows | dpr_prevW {len(S['dpr_prevW']):,} rows")
        report_source_grain(S)
    broken = [k for k, ok in S["source_usable"].items()
              if not ok and S["input_diagnostics"].set_index("source")
              .loc[k, "raw_rows"] > 0]
    if broken and RAISE_ON_UNUSABLE_SOURCE:
        raise ValueError(f"unusable source(s): {broken}. Set RAISE_ON_UNUSABLE_SOURCE=False "
                         f"to build anyway with those features left NaN.")

    w = S["dpr_W"]
    C = {
        "cat_levels": (pick_levels(w["donor_id"], w["project_category"])
                      if "project_category" in w.columns else []),
        "grade_levels": (pick_levels(w["donor_id"], w["project_grade"])
                        if "project_grade" in w.columns else []),
        "channel_levels": (pick_levels(w["donor_id"], referral_channel(w))
                          if not w.empty else []),
        "zip_state": zip_state_map(zip_acs),
    }
    if verbose:
        print(f"\nShare-vector levels (threshold: >={MIN_LEVEL_PREVALENCE:.0%} of donors)")
        specs = [("project_category", C["cat_levels"]), ("project_grade", C["grade_levels"])]
        for cname, kept in specs:
            if cname not in w.columns:
                continue
            rep = level_report(w["donor_id"], w[cname], kept, w.get("payment_amount"))
            folded = rep.loc[~rep["kept"]]
            print(f"  {cname}: kept {len(kept)} of {len(rep)}")
            if not folded.empty:
                print(f"    folded into _other  (check amount_share for anything "
                      f"rare-but-valuable):")
                print(folded[["donor_prevalence", "gift_share", "amount_share"]]
                      .to_string().replace("\n", "\n      "))
        if not w.empty:
            rep = level_report(w["donor_id"], referral_channel(w), C["channel_levels"],
                               w.get("payment_amount"))
            print(f"  referral_channel: kept {len(C['channel_levels'])} of {len(rep)}")
        print(f"  zip->state map: "
              f"{'available' if C['zip_state'] is not None else 'NOT AVAILABLE'}\n")

    # Collect blocks then concat once. Iterative `.join()` fragments the frame and emits a
    # PerformanceWarning at this column count.
    blocks, block_of, seen = [], {}, set()
    for name, fn in BLOCK_ORDER:
        blk = fn(S, C).reindex(S["universe"])
        dupes = sorted(set(blk.columns) & seen)
        if dupes:
            raise ValueError(f"block {name} would overwrite existing columns: {dupes}")
        seen.update(blk.columns)
        blocks.append(blk)
        block_of.update({c: name for c in blk.columns})
        if verbose:
            print(f"  {name:<20s} +{blk.shape[1]:>3d} cols -> {len(seen):>3d} total")

    # `.copy()` consolidates the block-wise inserts; without it pandas emits a
    # fragmentation PerformanceWarning at this column count.
    features = pd.concat(blocks, axis=1).copy()
    features.index.name = "donor_id"

    meta = assign_roles(list(features.columns))
    meta["block"] = meta["field"].map(block_of)
    meta["status"] = np.where(meta["field"].isin(BLOCKED_FIELDS), "BLOCKED", "ACTIVE")
    unsafe = {
        f for f, src in AS_OF_EXTRACT_SUSPECTS.items()
        if S["as_of_extract_columns"].get(src, False)}
    meta["snapshot_safe"] = ~meta["field"].isin(unsafe)
    meta["dtype"] = meta["field"].map(features.dtypes.astype("string").to_dict())
    nn = features.notna().mean()
    meta["fill_rate_all"] = meta["field"].map(nn.to_dict()).astype("float64").round(4)
    elig = features["cluster_eligible"].fillna(0) == 1
    nn_e = features.loc[elig].notna().mean()
    meta["fill_rate_eligible"] = meta["field"].map(nn_e.to_dict()).astype("float64").round(4)

    features = features.reset_index()
    meta = pd.concat([pd.DataFrame([{"field": "donor_id", "role": "KEY",
                                     "block": "00_population", "status": "ACTIVE",
                                     "dtype": "object", "fill_rate_all": 1.0,
                                     "fill_rate_eligible": 1.0, "snapshot_safe": True}]),
                      meta], ignore_index=True).drop_duplicates("field", keep="first")
    return features, meta


def cluster_matrix_fields(meta: pd.DataFrame, categorical: bool = False,
                          snapshot_safe_only: bool = False) -> list[str]:
    """Field list for the clustering matrix.

    `categorical=False` -> numeric CLUSTER fields (k-means path, one-hots included).
    `categorical=True`  -> adds the raw string CLUSTER_CAT fields and removes their
    one-hot counterparts (Gower / k-medoids path).
    `snapshot_safe_only=True` -> drops fields whose value reflects extract time rather than
    T. Leave it False for a segmentation at a fixed T; set it True before reusing these
    features to predict anything.
    """
    m = meta.loc[meta["status"] == "ACTIVE"]
    if snapshot_safe_only and "snapshot_safe" in m.columns:
        m = m.loc[m["snapshot_safe"]]
    num = m.loc[m["role"] == "CLUSTER", "field"].tolist()
    if not categorical:
        return num
    cats = m.loc[m["role"] == "CLUSTER_CAT", "field"].tolist()
    onehot = {c for c in num for cat in cats if c.startswith(cat + "_")}
    return [c for c in num if c not in onehot] + cats

## Section 20 — Run

`load_source` reads only the declared columns and caches a binary copy, invalidated by the
CSV's modification time so a re-export is picked up automatically. `prepare` then restricts
every event table to the donor universe before any feature work — loss-free, because the
universe is defined from `dpr` and `monthly` alone, so email, site and share can never add
a donor. On a 50k-donor universe against 84.5M email rows spanning far more donors, that is
the difference between a one-off run and an iterable one.

In [21]:
_cache = CACHE_DIR if USE_SOURCE_CACHE else None
df_dpr = load_source(LOAD_PATH / FILES["dpr"], "dpr", cache_dir=_cache)
df_email = load_source(LOAD_PATH / FILES["email"], "email", cache_dir=_cache)
df_site = load_source(LOAD_PATH / FILES["site"], "site", cache_dir=_cache)
df_monthly = load_source(LOAD_PATH / FILES["monthly"], "monthly", cache_dir=_cache)
df_share = load_source(LOAD_PATH / FILES["share"], "share", cache_dir=_cache)
df_project_dates = (load_source(LOAD_PATH / FILES["project_dates"], "project_dates",
                                restrict_columns=False, cache_dir=_cache)
                    if LOAD_PROJECT_DATES else None)
df_zip_acs = load_source(LOAD_PATH / FILES["zip_acs"], "zip_acs",
                         restrict_columns=False, cache_dir=_cache)

for n, d in [("dpr", df_dpr), ("email", df_email), ("site", df_site), ("monthly", df_monthly),
             ("share", df_share), ("project_dates", df_project_dates), ("zip_acs", df_zip_acs)]:
    print(f"{n:14s} " + ("not loaded (LOAD_PROJECT_DATES=False)" if d is None
                         else f"{d.shape[0]:>9,} rows x {d.shape[1]:>3} cols"))

  dpr            not in this file: ['project_fully_funded']
  email            84,533,921 rows x  5 cols from cache email.v6.1a9642de68.parquet
  site             18,663,154 rows x  8 cols from cache site.v6.57400ca468.parquet
  monthly           1,165,365 rows x  6 cols from cache monthly.v6.89c142b13c.parquet
  share             1,331,919 rows x  3 cols from cache share.v6.a6b2fe9a4b.parquet
  zip_acs              33,772 rows x 10 cols from cache zip_acs.v6.9387d7c4a1.parquet
dpr            12,285,666 rows x  40 cols
email          84,533,921 rows x   5 cols
site           18,663,154 rows x   8 cols
monthly        1,165,365 rows x   6 cols
share          1,331,919 rows x   3 cols
project_dates  not loaded (LOAD_PROJECT_DATES=False)
zip_acs           33,772 rows x  10 cols


In [22]:
features, field_meta = build_clustering_features(
    dpr=df_dpr, email=df_email, site=df_site, monthly=df_monthly, share=df_share,
    project_dates=df_project_dates, zip_acs=df_zip_acs, T=T, window_months=WINDOW_MONTHS)

print(f"\nfeatures: {features.shape[0]:,} rows x {features.shape[1]:,} cols")
print(field_meta["role"].value_counts().to_string())

T=2026-08-01  W=[2024-08-01, 2026-08-01)
universe: 918,523 donors | dpr_W 2,751,764 rows | dpr_prevW 1,527,943 rows

Date columns  (pop = share of rows populated; ok = share of those that are plausible dates)
  dpr      payment_date                           pop 100.0% (12,285,666/12,285,666) | ok 100.0% | to_datetime [gates presence]
  dpr      project_expiration_date                pop  85.5% (10,501,812/12,285,666) | ok 100.0% | to_datetime
  monthly  monthly_subscription_payment_date      pop 100.0% (1,165,365/1,165,365) | ok 100.0% | to_datetime [gates presence]
  monthly  monthly_subscription_joined_date       pop 100.0% (1,165,365/1,165,365) | ok 100.0% | to_datetime [gates presence]
  monthly  monthly_subscription_retired_date      pop  86.2% (1,004,942/1,165,365) | ok 100.0% | to_datetime
  site     activity_date                          pop 100.0% (10,309,590/10,309,590) | ok 100.0% | to_datetime [gates presence]
  email    email_month_start                      pop 100.0% (2

## Section 21 — QA

Eight checks, each targeting a specific way a clustering matrix goes wrong, plus three
helpers that turn the findings into actions: `degenerate_fields` (constant columns),
`redundant_fields` (near-duplicate pairs, which constant-detection cannot catch), and
`apply_coverage_floor` (weak-evidence shares). Read these before running k-means; they are
cheaper than diagnosing a bad solution afterwards.

In [23]:
def degenerate_fields(features: pd.DataFrame, meta: pd.DataFrame) -> list[str]:
    """CLUSTER fields that are constant or empty among eligible donors. Drop these before
    scaling -- a zero-variance column makes StandardScaler divide by zero."""
    f = features.set_index("donor_id") if "donor_id" in features.columns else features
    elig = f["cluster_eligible"].fillna(0) == 1
    cols = cluster_matrix_fields(meta)
    nun = f.loc[elig, cols].nunique(dropna=True)
    return sorted(nun[nun <= 1].index)


def redundant_fields(features: pd.DataFrame, meta: pd.DataFrame,
                     threshold: float = 0.98, categorical: bool = False) -> pd.DataFrame:
    """Near-duplicate CLUSTER field pairs, as a frame of `keep` / `drop` / `r`.

    `degenerate_fields` cannot catch these: an exact complement such as
    `share_gifts_repeat_teacher_12m` vs `share_gifts_new_teacher_12m` has full variance and
    is perfectly informative -- it is only redundant *alongside its twin*. Both halves are
    built deliberately so either can be chosen, but only one belongs in the matrix.

    Returns rather than drops, because for an exact complement the choice of survivor is
    arbitrary (here, alphabetically first) and interpretability may favour the other.
    """
    f = features.set_index("donor_id") if "donor_id" in features.columns else features
    elig = f["cluster_eligible"].fillna(0) == 1
    cols = sorted(c for c in cluster_matrix_fields(meta, categorical)
                  if pd.api.types.is_numeric_dtype(f[c]))
    X = f.loc[elig, cols]
    usable = [c for c in cols if X[c].notna().sum() > 30 and X[c].nunique(dropna=True) > 1]
    if len(usable) < 2:
        return pd.DataFrame(columns=["keep", "drop", "r"])

    corr = X[usable].corr(numeric_only=True).abs()
    dropped, rows = set(), []
    for i, a in enumerate(usable):
        if a in dropped:
            continue
        for b in usable[i + 1:]:
            if b in dropped:
                continue
            r = corr.at[a, b]
            if pd.notna(r) and r >= threshold:
                dropped.add(b)
                rows.append({"keep": a, "drop": b, "r": round(float(r), 4)})
    return pd.DataFrame(rows).sort_values("r", ascending=False).reset_index(drop=True)


# Which fields each coverage diagnostic gates. Declared explicitly rather than pattern-
# guessed, so adding a location or cost field does not silently escape the floor.
COVERAGE_GATES = {
    "coverage_distance_12m": (r"^(median|mean|min|max|cv)_distance_mi_12m$"
                             r"|^share_gifts_(within_\d+mi|over_500mi)_12m$"),
    "coverage_zip_12m": r"^share_gifts_same_zip3?_12m$|^n_unique_school_zips_12m$",
    "coverage_state_12m": r"^share_gifts_same_state_12m$|^n_unique_school_states_12m$",
    "coverage_project_cost_12m": (r"_gift_to_project_cost_ratio_12m$"
                                  r"|^share_gifts_(over_half|full)_project_cost_12m$"),
    "coverage_project_expiration_12m": (r"^funding_cycle_position_mean_12m$"
                                        r"|^share_gifts_late_cycle_12m$"
                                        r"|^days_gift_to_expiration_median_12m$"
                                        r"|^share_gifts_expiring_soon_12m$"),
    "coverage_school_equity_flags_12m":
        r"^share_gifts_to_(low_income_schools|historically_underrepresented_race_schools"
        r"|underserved_rural_schools|low_income_plus_historically_underrepresented_race_schools)_12m$",
    "decision_speed_measurable_12m":
        r"^(project_page|teacher_page|search|cart|site_days)_pre_gift_mean_12m$",
}


def apply_coverage_floor(features: pd.DataFrame, floor: float = 0.5
                         ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Mask gated fields to NaN where their coverage falls below `floor`.

    A proximity share computed over 2 of 30 measurable gifts is not the same evidence as
    the same number computed over 30 of 30, but a distance metric cannot tell them apart.
    Masking hands the decision to the NaN policy instead of silently averaging weak
    evidence in. Returns (masked copy, report).
    """
    out = features.copy()
    rows = []
    for cov, pattern in COVERAGE_GATES.items():
        if cov not in out.columns:
            continue
        gated = [c for c in out.columns if re.search(pattern, c)]
        if not gated:
            continue
        below = out[cov].notna() & (out[cov] < floor)
        out.loc[below, gated] = np.nan
        rows.append({"coverage_field": cov, "n_gated_fields": len(gated),
                     "donors_masked": int(below.sum()),
                     "pct_donors_masked": round(float(below.mean()), 4)})
    return out, pd.DataFrame(rows)


def qa_report(features: pd.DataFrame, meta: pd.DataFrame, top_n: int = 12) -> None:
    f = features.set_index("donor_id")
    elig = f["cluster_eligible"].fillna(0) == 1
    num = cluster_matrix_fields(meta, categorical=False)
    cats = meta.loc[(meta["role"] == "CLUSTER_CAT")
                    & (meta["status"] == "ACTIVE"), "field"].tolist()
    blocked = sorted(meta.loc[meta["status"] == "BLOCKED", "field"])
    X = f.loc[elig, num]

    print(f"eligible donors: {int(elig.sum()):,} of {len(f):,}")
    print(f"k-means matrix (role=CLUSTER, numeric): {len(num)} fields")
    print(f"Gower alternative (role=CLUSTER_CAT):   {len(cats)} fields -> {cats}")
    print(f"blocked feature fields:                 {len(blocked)}")
    if "snapshot_safe" in meta.columns:
        unsafe = meta.loc[~meta["snapshot_safe"], "field"].tolist()
        print(f"snapshot-unsafe (extract-time values):  {len(unsafe)}"
              + (f" -> {unsafe}" if unsafe else ""))
        print("  ^ harmless at a fixed T; pass snapshot_safe_only=True before predicting")
    print()

    print("1. UNBOUNDED VALUES  (was the 1e-6 denominator pattern; expect none)")
    mx = X.abs().max().sort_values(ascending=False)
    bad = mx[mx > 1e4]
    print("   none above 1e4" if bad.empty else bad.head(top_n).to_string())

    print("\n2. CONSTANT / EMPTY  (zero variance breaks scaling -- drop before fitting)")
    const = degenerate_fields(features, meta)
    if not const:
        print("   none")
    else:
        print(f"   {len(const)} fields; drop via degenerate_fields(features, field_meta)")
        for c in const[:top_n]:
            print(f"     {c}")

    print("\n3. LOW FILL RATE among eligible donors (<50%)")
    fr = X.notna().mean().sort_values()
    low = fr[fr < 0.5]
    print("   none" if low.empty else low.head(top_n).round(3).to_string())

    print("\n4. NEAR-DUPLICATE PAIRS  (|r| >= 0.98; exact complements appear here at 1.000)")
    red = redundant_fields(features, meta, threshold=0.98)
    if red.empty:
        print("   none")
    else:
        print(f"   {len(red)} pairs; review and drop via "
              f"redundant_fields(features, field_meta)")
        print("   " + red.head(top_n).to_string(index=False).replace("\n", "\n   "))

    print("\n5. SHARE FAMILIES — rows should sum to 1.0 where populated")
    fams = {"season": r"^share_gifts_(year_end|giving_tuesday|december_other|back_to_school"
                      r"|teacher_appreciation|summer|other)",
            "category": r"^share_count_category_", "grade": r"^share_count_grade_",
            "position": r"^share_gifts_(sole_funder|first_money_in|closed_project|mid_funding)",
            "month": r"^month_share_count_", "device": r"^device_share_"}
    for fam, pat in fams.items():
        cols = [c for c in f.columns if re.search(pat, c)]
        if not cols:
            continue
        s = f.loc[elig, cols].sum(axis=1)
        s = s[f.loc[elig, cols].notna().any(axis=1)]
        ok = bool(s.empty) or bool(np.allclose(s.dropna(), 1.0, atol=1e-6))
        print(f"   {fam:<10s} {len(cols):>2d} cols  sums_to_1={ok}"
              f"  min={s.min():.4f} max={s.max():.4f}" if not s.empty
              else f"   {fam:<10s} {len(cols):>2d} cols  (no populated rows)")

    print("\n6. PERIOD-COUNT BOUNDS")
    # The ceiling is NOT `WINDOW_MONTHS`. A 12-month window opening mid-month touches 13
    # calendar months, so daily-grain counts can legitimately reach 13. Monthly-grain
    # sources (email, share) have their dates snapped to the 1st before the window filter,
    # so they top out one lower. Comparing both against 12 produced a false alarm.
    W0 = pd.to_datetime(f["window_start"].iloc[0])
    T0 = pd.to_datetime(f["snapshot_date"].iloc[0])
    ceil = {"daily": calendar_months_touched(W0, T0),
            "monthly_grain": month_starts_in(W0, T0)}
    print(f"   window [{W0:%Y-%m-%d}, {T0:%Y-%m-%d}) touches "
          f"{ceil['daily']} calendar months; monthly-grain sources see "
          f"{ceil['monthly_grain']}")
    if ceil["daily"] != ceil["monthly_grain"]:
        print(f"   note: the window opens mid-month, so gift-based and email/share-based "
              f"month\n         counts have different ceilings. Setting T to a month start "
              f"aligns them.")
    mcols = [c for c in f.columns if re.search(r"(active_months|months_with_\w+)_12m$", c)]
    unmapped = [c for c in mcols if c not in MONTH_COUNT_BASIS]
    for c in mcols:
        basis = MONTH_COUNT_BASIS.get(c)
        v = f[c].dropna()
        mx = float(v.max()) if not v.empty else 0.0
        if basis is None:
            print(f"   {c:<40s} max={mx:>6.1f}  basis=UNDECLARED")
            continue
        lim = ceil[basis]
        print(f"   {c:<40s} max={mx:>6.1f}  limit={lim:>3d} ({basis})  ok={mx <= lim}")
    if unmapped:
        print(f"   ^ add {unmapped} to MONTH_COUNT_BASIS so they get the right ceiling")

    print("\n7. PROPORTION BOUNDS  (every `share_*` field must lie in [0, 1])")
    scols = [c for c in f.columns if c.startswith("share_")
             and pd.api.types.is_numeric_dtype(f[c])]
    off = {c: (float(f[c].min()), float(f[c].max()))
           for c in scols if f[c].notna().any()
           and not f[c].dropna().between(-1e-9, 1 + 1e-9).all()}
    if not off:
        print(f"   all {len(scols)} share_* fields within [0, 1]")
    else:
        for c, (lo_, hi_) in list(off.items())[:top_n]:
            print(f"   {c:<46s} [{lo_:.3f}, {hi_:.3f}]")
        print("   ^ a share outside [0,1] means the denominator is wrong "
              "or the name is misleading")

    print("\n8. COVERAGE  (denominators behind the observed-basis shares)")
    ccols = sorted(c for c in f.columns if c.startswith("coverage_"))
    if not ccols:
        print("   no coverage fields")
    else:
        for c in ccols:
            v = f.loc[elig, c].dropna()
            if v.empty:
                print(f"   {c:<32s} (no data)")
            else:
                print(f"   {c:<32s} mean={v.mean():.3f}  "
                      f"donors fully covered={float((v >= 1.0).mean()):.3f}")
        print("   ^ mask weak-evidence shares with "
              "apply_coverage_floor(features, floor=0.5)")

    print("\n9. MONTHLY CONSISTENCY")
    if {"monthly_payments_12m", "monthly_active_months_12m"} <= set(f.columns):
        p = f["monthly_payments_12m"]
        m = f["monthly_active_months_12m"]
        bad = p.notna() & m.notna() & (m > p)
        print("   active months <= payments for every donor"
              if not bad.any() else
              f"   CHECK: {int(bad.sum()):,} donors have more active months than payments")
    else:
        print("   monthly payment-count fields not present")

    if {"has_monthly_payment_12m", "n_recurring_donation_ids"} <= set(f.columns):
        paid = f["has_monthly_payment_12m"].fillna(0) > 0
        rel = f["n_recurring_donation_ids"].fillna(0)
        bad = paid & (rel < 1)
        print("   every 12m monthly payer has an observed recurring relationship"
              if not bad.any() else
              f"   CHECK: {int(bad.sum()):,} monthly payers have no relationship id")

    if {"is_monthly_donor_current", "n_active_recurring_donation_ids",
        "days_since_last_monthly_payment"} <= set(f.columns):
        cur = f["is_monthly_donor_current"].fillna(0) > 0
        active_n = f["n_active_recurring_donation_ids"].fillna(0)
        recency = f["days_since_last_monthly_payment"]
        bad_binary = cur.ne(active_n > 0)
        bad_recency = cur & (recency.isna() | (recency > MONTHLY_ACTIVE_LOOKBACK_DAYS))
        if not bad_binary.any() and not bad_recency.any():
            print(f"   monthly active status matches <= {MONTHLY_ACTIVE_LOOKBACK_DAYS}d payment recency")
        else:
            print(f"   CHECK: active-count/binary mismatches={int(bad_binary.sum()):,}; "
                  f"active donors outside recency window={int(bad_recency.sum()):,}")

    unmatched = [c for c in meta["field"] if c not in f.columns and c != "donor_id"]
    if unmatched:
        print(f"\nWARNING: meta references missing fields: {unmatched}")


qa_report(features, field_meta)

eligible donors: 918,263 of 918,523
k-means matrix (role=CLUSTER, numeric): 199 fields
Gower alternative (role=CLUSTER_CAT):   1 fields -> ['entry_channel']
blocked feature fields:                 0
snapshot-unsafe (extract-time values):  3 -> ['mean_teacher_lifetime_projects_fully_funded_12m', 'mean_teacher_lifetime_donations_12m', 'latest_teacher_lifetime_projects_fully_funded']
  ^ harmless at a fixed T; pass snapshot_safe_only=True before predicting

1. UNBOUNDED VALUES  (was the 1e-6 denominator pattern; expect none)
median_project_total_cost_12m          81353.84
mean_project_total_cost_12m            81353.84
mean_teacher_lifetime_donations_12m    13684.00

2. CONSTANT / EMPTY  (zero variance breaks scaling -- drop before fitting)
   9 fields; drop via degenerate_fields(features, field_meta)
     device_share_other_12m
     device_share_unknown_12m
     is_sharer_12m
     n_unique_school_states_12m
     share_count_grade_other_12m
     share_gifts_channel_unknown_12m
     share_

In [24]:
print("Fields by block and role\n")
print(field_meta.groupby(["block", "role"], observed=True).size()
      .unstack(fill_value=0).to_string())

Fields by block and role

role               CLUSTER  CLUSTER_CAT  DIAG  KEY  PROFILE
block                                                      
00_population            0            0     8    3        0
01_scaffold              2            0     1    0        1
02_value                 0            0     0    0       19
03_rhythm                9            0     0    0        0
04_velocity              5            0     0    0        0
05_tenure                4            0     0    2        1
06_entry                19            1     2    0        0
07_monthly              12            0     0    0        2
08_timing               12            0     0    0       20
09_gift_size            13            0     1    0        0
10_project_state        15            0     1    0        2
11_location             16            0     3    0        0
12_topic                29            0     2    0       13
12b_loyalty             20            0     0    0        0
13_payment_mix

## Section 22 — Write

In [25]:
WRITE_PATH.mkdir(parents=True, exist_ok=True)
feat_path = WRITE_PATH / f"clustering_features_{OUT_TAG}.csv"
meta_path = WRITE_PATH / f"clustering_field_meta_{OUT_TAG}.csv"
features.to_csv(feat_path, index=False)
field_meta.to_csv(meta_path, index=False)
print(f"wrote {feat_path}  ({features.shape[0]:,} x {features.shape[1]:,})")
print(f"wrote {meta_path}  ({field_meta.shape[0]:,} fields)")

wrote /Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data/clustering_features_20260801_w24m.csv  (918,523 x 331)
wrote /Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data/clustering_field_meta_20260801_w24m.csv  (331 fields)


## Section 23 — Pre-clustering transform checklist

Deliberately **not** applied here — feature construction and modelling decisions stay
separate. Do this in the clustering script:

1. **Filter** to `cluster_eligible == 1`. Carry monthly-only donors as a declared segment.
2. **Select** by role, then drop degenerate columns:

   ```python
   cols = cluster_matrix_fields(field_meta)
   cols = cluster_matrix_fields(field_meta, categorical=True)
   cols = cluster_matrix_fields(field_meta, snapshot_safe_only=True)
   cols = [c for c in cols if c not in degenerate_fields(features, field_meta)]
   cols = [c for c in cols
           if c not in set(redundant_fields(features, field_meta)["drop"])]
   ```

   Review exact/near-duplicate pairs before dropping them; the statistically equivalent
   survivor is not always the most interpretable one.

3. **Monthly is a required source contract, not an inferred grain.** The feature build stops
   if the payment-row / recurring-relationship contract fails. For a donor with no recurring
   payments, counts and amounts are true zeroes; characteristics of nonexistent behavior
   (recency, median amount, streak, join timing) remain NaN.
4. **Current monthly status is payment-derived and snapshot-safe.**
   A recurring relationship is behaviorally active when its last observed payment before `T`
   is within 30 days. `is_monthly_donor_current` is 1 when any recurring relationship meets
   that rule; `n_active_recurring_donation_ids` counts how many do.
5. **Optionally mask weak-evidence shares** before imputation:

   ```python
   features, cov_report = apply_coverage_floor(features, floor=0.5)
   ```

6. **Log1p** the skewed continuous fields as appropriate: `tenure_days`,
   `median_distance_mi_12m`, project-cost fields, visit duration, gift span, and teacher
   lifetime counters are the obvious candidates.
7. **Normalised entropy** (`*_norm_12m`) uses the donor's achievable entropy ceiling and is
   therefore comparable across gift frequencies.
8. **NaN policy, per family, deliberately:**
   - entropy / CV / gap fields are NaN by design at low counts;
   - monthly counts/amounts are already zero for donors with no monthly behavior, while
     monthly characteristics remain NaN when there is nothing to characterize;
   - site / email / decision-speed require explicit coverage-aware treatment.
9. **Share-block scaling.** Scale each share family together or leave it on 0–1; do not
   independently z-score rare share levels into equal influence.
10. **Collinearity.** Each complete share family has K−1 effective dimensions.
11. **Sensitivity runs.** The solution should survive excluding low-count shares,
   `latest_*`, and the decision-speed block.

### Snapshot safety

`field_meta.snapshot_safe` is False only for fields whose value reflects extract time rather
than the state at `T`. All monthly outputs are snapshot-safe: payment rows are filtered to
`< T`, and current monthly activity is defined from the most recent observed payment before
`T` rather than from the source's extract-time active flag.

`monthly_longest_streak_months` is derived from distinct observed payment months through `T`;
the supplied `monthly_subscription_longest_streak` rollup is intentionally not used by
the feature builder. `monthly_subscription_retired_date` is also QA-only because the
explored source contains material payments after recorded retirement dates.

### Carried caveats

- `entry_channel` is unreliable where the project extract truncates history. Cross-check
  `entry_donation_n`; values other than 1 mean the true first project gift is outside the
  extract.
- Paid media is not separable from organic with the current source whitelist.
- No project/fund/designation key is available; four project-state fields remain blocked.
- Decision speed is donor-day aggregated, so it has no within-session ordering.
- Site visit-basis fields aggregate over sessions when `session_id` exists and donor-days
  otherwise; `site_unit_is_session` records which.
- Coverage fields (distance, ZIP, state, project cost) should gate weak-evidence shares.
- Share-vector levels are thresholded on donor prevalence, not gift prevalence; review the
  printed folded-level report for rare but valuable behavior.